In [10]:
# ============================================================
# MODEL 3 — DATASET CONFIGURATION
# ============================================================

from pathlib import Path
import json
import random

import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader


# ============================================================
# DEBUG SWITCH
# ============================================================

DEBUG = False

# True  -> small dataset for testing/debugging
# False -> full 200M-token dataset


# ============================================================
# REPRODUCIBILITY
# ============================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)


# ============================================================
# TOKENIZER / MODEL SETTINGS
# ============================================================

VOCAB_SIZE = 32004
BLOCK_SIZE = 2048
TOKEN_DTYPE = np.uint16


# ============================================================
# TOKEN BUDGET
# ============================================================

FULL_TOTAL_TOKENS = 200_000_000

DEBUG_TOTAL_TOKENS = 2_000_000


TOTAL_TOKEN_BUDGET = (
    DEBUG_TOTAL_TOKENS
    if DEBUG
    else FULL_TOTAL_TOKENS
)


# ============================================================
# SOURCE RATIOS
# ============================================================

SOURCE_RATIOS = {
    "acts": 15,
    "high_courts": 35,
    "supreme_court": 25,
    "tribunals": 25,
}


# ============================================================
# TRAIN / TEST SPLIT
# ============================================================

TRAIN_RATIO = 0.98
TEST_RATIO = 0.02


# ============================================================
# DATA PATH
# ============================================================

TOKENIZED_DIR = Path(
    "/kaggle/input/notebooks/ritikguptatab/updated-tokenize-data/legal_tokenized_100m"
)


SOURCE_FILES = {
    "acts":
        TOKENIZED_DIR / "acts_100m.bin",

    "high_courts":
        TOKENIZED_DIR / "high_courts_100m.bin",

    "supreme_court":
        TOKENIZED_DIR / "supreme_court_100m.bin",

    "tribunals":
        TOKENIZED_DIR / "tribunals_100m.bin",
}


# ============================================================
# OUTPUT
# ============================================================

RUN_NAME = (
    "debug"
    if DEBUG
    else "full"
)

OUTPUT_DIR = Path(
    "/kaggle/working/model3_data"
) / RUN_NAME

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# DATALOADER
# ============================================================

NUM_WORKERS = 2
PIN_MEMORY = torch.cuda.is_available()
from torch.utils.data import DataLoader


BATCH_SIZE = 1
MICRO_BATCH_SIZE = 1

TRAIN_NUM_WORKERS = 2
VAL_NUM_WORKERS = 2

print("=" * 70)
print("MODEL 3 DATASET CONFIG")
print("=" * 70)

print("DEBUG              :", DEBUG)
print("TOTAL TOKENS      :", f"{TOTAL_TOKEN_BUDGET:,}")
print("BLOCK SIZE        :", BLOCK_SIZE)
print("VOCAB SIZE        :", VOCAB_SIZE)
print("OUTPUT            :", OUTPUT_DIR)

MODEL 3 DATASET CONFIG
DEBUG              : False
TOTAL TOKENS      : 200,000,000
BLOCK SIZE        : 2048
VOCAB SIZE        : 32004
OUTPUT            : /kaggle/working/model3_data/full


In [11]:
# ============================================================
# VERIFY SOURCE FILES
# ============================================================

for source, path in SOURCE_FILES.items():

    print(
        f"{source:15s} -> {path}"
    )

    if not path.exists():

        raise FileNotFoundError(
            f"Missing source file:\n{path}"
        )

print("\n✓ All source files found.")

acts            -> /kaggle/input/notebooks/ritikguptatab/updated-tokenize-data/legal_tokenized_100m/acts_100m.bin
high_courts     -> /kaggle/input/notebooks/ritikguptatab/updated-tokenize-data/legal_tokenized_100m/high_courts_100m.bin
supreme_court   -> /kaggle/input/notebooks/ritikguptatab/updated-tokenize-data/legal_tokenized_100m/supreme_court_100m.bin
tribunals       -> /kaggle/input/notebooks/ritikguptatab/updated-tokenize-data/legal_tokenized_100m/tribunals_100m.bin

✓ All source files found.


In [12]:
import shutil

source_dir = "/kaggle/input/models/ritikguptatab/proj-cap-p-1-archi-4-12k/pytorch/default/4"
destination_dir = "/kaggle/working"

# Merges the replica into the existing directory, overwriting matching files
shutil.copytree(source_dir, destination_dir, dirs_exist_ok=True)


'/kaggle/working'

In [13]:
# ============================================================
# MEMORY-MAPPED TOKEN SOURCE
# ============================================================

class TokenFile:

    def __init__(
        self,
        path,
        dtype=np.uint16,
    ):

        self.path = Path(path)
        self.dtype = np.dtype(dtype)

        file_size = (
            self.path.stat().st_size
        )

        if (
            file_size
            % self.dtype.itemsize
            != 0
        ):

            raise ValueError(
                f"Invalid token file: {self.path}"
            )

        self.n_tokens = (
            file_size
            // self.dtype.itemsize
        )

        self.data = np.memmap(
            self.path,
            dtype=self.dtype,
            mode="r",
            shape=(self.n_tokens,),
        )


    def read(
        self,
        start,
        count,
    ):

        return np.asarray(
            self.data[
                start:start + count
            ]
        )

In [14]:
# ============================================================
# LOAD ALL FOUR SOURCES
# ============================================================

SOURCE_DATA = {}

for source, path in SOURCE_FILES.items():

    SOURCE_DATA[source] = TokenFile(
        path,
        dtype=TOKEN_DTYPE,
    )

    available = (
        SOURCE_DATA[source].n_tokens
    )

    print(
        f"{source:15s} "
        f"{available:,} tokens"
    )


print("\n✓ All sources loaded using mmap.")

acts            256,521,619 tokens
high_courts     242,356,017 tokens
supreme_court   241,501,699 tokens
tribunals       241,779,387 tokens

✓ All sources loaded using mmap.


In [15]:
# ============================================================
# SOURCE TOKEN BUDGET
# ============================================================

SOURCE_BUDGET = {}

for source, ratio in SOURCE_RATIOS.items():

    SOURCE_BUDGET[source] = int(
        TOTAL_TOKEN_BUDGET
        * ratio
        / 100
    )


print("=" * 70)
print("SOURCE BUDGET")
print("=" * 70)

for source in SOURCE_BUDGET:

    print(
        f"{source:15s}"
        f"{SOURCE_BUDGET[source]:>15,}"
    )


assert (
    sum(SOURCE_BUDGET.values())
    == TOTAL_TOKEN_BUDGET
)

SOURCE BUDGET
acts                30,000,000
high_courts         70,000,000
supreme_court       50,000,000
tribunals           50,000,000


In [16]:
# ============================================================
# 98 : 2 SPLIT
# ============================================================

TRAIN_BUDGET = {}
TEST_BUDGET = {}

for source in SOURCE_BUDGET:

    total = SOURCE_BUDGET[source]

    train_tokens = int(
        total * TRAIN_RATIO
    )

    test_tokens = (
        total - train_tokens
    )

    TRAIN_BUDGET[source] = (
        train_tokens
    )

    TEST_BUDGET[source] = (
        test_tokens
    )


TOTAL_TRAIN_TOKENS = sum(
    TRAIN_BUDGET.values()
)

TOTAL_TEST_TOKENS = sum(
    TEST_BUDGET.values()
)


print("=" * 70)
print("TRAIN / TEST SPLIT")
print("=" * 70)

for source in SOURCE_BUDGET:

    print(
        f"{source:15s}"
        f"total={SOURCE_BUDGET[source]:>12,} "
        f"train={TRAIN_BUDGET[source]:>12,} "
        f"test={TEST_BUDGET[source]:>10,}"
    )


print("-" * 70)

print(
    "TRAIN:",
    f"{TOTAL_TRAIN_TOKENS:,}"
)

print(
    "TEST :",
    f"{TOTAL_TEST_TOKENS:,}"
)

TRAIN / TEST SPLIT
acts           total=  30,000,000 train=  29,400,000 test=   600,000
high_courts    total=  70,000,000 train=  68,600,000 test= 1,400,000
supreme_court  total=  50,000,000 train=  49,000,000 test= 1,000,000
tribunals      total=  50,000,000 train=  49,000,000 test= 1,000,000
----------------------------------------------------------------------
TRAIN: 196,000,000
TEST : 4,000,000


In [17]:
# ============================================================
# CREATE MERGED DATASETS
# ============================================================

TRAIN_BIN = (
    OUTPUT_DIR / "train.bin"
)

TEST_BIN = (
    OUTPUT_DIR / "test.bin"
)


def create_empty_bin(
    path,
    num_tokens,
):

    with open(path, "wb") as f:

        f.truncate(
            num_tokens
            * np.dtype(
                TOKEN_DTYPE
            ).itemsize
        )


create_empty_bin(
    TRAIN_BIN,
    TOTAL_TRAIN_TOKENS,
)

create_empty_bin(
    TEST_BIN,
    TOTAL_TEST_TOKENS,
)


train_mm = np.memmap(
    TRAIN_BIN,
    dtype=TOKEN_DTYPE,
    mode="r+",
    shape=(TOTAL_TRAIN_TOKENS,),
)

test_mm = np.memmap(
    TEST_BIN,
    dtype=TOKEN_DTYPE,
    mode="r+",
    shape=(TOTAL_TEST_TOKENS,),
)


train_offset = 0
test_offset = 0


# ============================================================
# COPY EACH SOURCE
# ============================================================

for source in SOURCE_RATIOS:

    source_data = SOURCE_DATA[
        source
    ]

    train_count = TRAIN_BUDGET[
        source
    ]

    test_count = TEST_BUDGET[
        source
    ]


    # --------------------------------------------------------
    # TRAIN PART
    # --------------------------------------------------------

    train_mm[
        train_offset:
        train_offset + train_count
    ] = source_data.read(
        0,
        train_count,
    )


    # --------------------------------------------------------
    # TEST PART
    # --------------------------------------------------------

    test_mm[
        test_offset:
        test_offset + test_count
    ] = source_data.read(
        train_count,
        test_count,
    )


    train_offset += train_count
    test_offset += test_count


    print(
        f"Processed {source}"
    )


train_mm.flush()
test_mm.flush()

del train_mm
del test_mm


print("\n✓ Merged dataset created.")

Processed acts
Processed high_courts
Processed supreme_court
Processed tribunals

✓ Merged dataset created.


In [18]:
# ============================================================
# CELL 8 — CAUSAL LM DATASET
# ============================================================

class CausalLMDataset(
    Dataset
):
    """
    Memory-mapped dataset for next-token prediction.

    For every sample:

        x = [t0, t1, ..., t(B-1)]

        y = [t1, t2, ..., tB]

    where B = block_size.
    """

    def __init__(
        self,
        bin_path,
        num_tokens,
        block_size=2048,
    ):

        self.bin_path = Path(
            bin_path
        )

        self.block_size = int(
            block_size
        )

        self.num_tokens = int(
            num_tokens
        )

        # ----------------------------------------------------
        # Memory-map token file
        # ----------------------------------------------------

        self.tokens = np.memmap(
            self.bin_path,
            dtype=TOKEN_DTYPE,
            mode="r",
            shape=(self.num_tokens,),
        )

        # ----------------------------------------------------
        # Number of complete non-overlapping LM windows
        #
        # Need B+1 tokens because target is shifted by one.
        # ----------------------------------------------------

        usable_starts = (
            self.num_tokens
            - self.block_size
        )

        self.num_examples = max(
            0,
            (
                num_tokens
                - (block_size + 1)
            ) // block_size
            + 1
        )


    def __len__(self):

        return self.num_examples


    def __getitem__(
        self,
        idx,
    ):

        if idx < 0 or idx >= self.num_examples:

            raise IndexError(
                f"Index {idx} out of range "
                f"for dataset of size {self.num_examples}"
            )

        # ----------------------------------------------------
        # Start of this non-overlapping window
        # ----------------------------------------------------

        start = (
            idx
            * self.block_size
        )

        # ----------------------------------------------------
        # Need block_size + 1 tokens
        # ----------------------------------------------------

        chunk = np.asarray(
            self.tokens[
                start:
                start
                + self.block_size
                + 1
            ],
            dtype=np.int64,
        )

        # ----------------------------------------------------
        # Input / target shift
        # ----------------------------------------------------

        x = torch.from_numpy(
            chunk[:-1]
        )

        y = torch.from_numpy(
            chunk[1:]
        )

        return x, y

In [19]:
# ============================================================
# ARCHITECTURE 4 — HYBRID PREFIX ENCODER LANGUAGE MODEL
# ============================================================

from dataclasses import dataclass
import math
import torch
import torch.nn as nn
import torch.nn.functional as F


@dataclass
class HybridLMConfig:

    # --------------------------------------------------------
    # Vocabulary / sequence
    # --------------------------------------------------------

    vocab_size: int = VOCAB_SIZE
    block_size: int = BLOCK_SIZE

    # --------------------------------------------------------
    # Transformer dimensions
    # --------------------------------------------------------

    d_model: int = 768
    n_layers: int = 16
    n_heads: int = 12

    # --------------------------------------------------------
    # Feed-forward network
    # --------------------------------------------------------

    ffn_dim: int = 2048

    # --------------------------------------------------------
    # Hybrid LM
    # --------------------------------------------------------

    prefix_length: int = 512

    # --------------------------------------------------------
    # Regularization
    # --------------------------------------------------------

    dropout: float = 0.0

    # --------------------------------------------------------
    # Normalization
    # --------------------------------------------------------

    norm_eps: float = 1e-5

    # --------------------------------------------------------
    # Linear layers
    # --------------------------------------------------------

    bias: bool = False


CONFIG = HybridLMConfig()


print("=" * 70)
print("ARCHITECTURE 4 — HYBRID PREFIX ENCODER LANGUAGE MODEL")
print("=" * 70)

print("Vocabulary size :", CONFIG.vocab_size)
print("Block size      :", CONFIG.block_size)
print("d_model         :", CONFIG.d_model)
print("Layers          :", CONFIG.n_layers)
print("Heads           :", CONFIG.n_heads)
print("Head dimension  :", CONFIG.d_model // CONFIG.n_heads)
print("FFN dimension   :", CONFIG.ffn_dim)
print("Prefix length   :", CONFIG.prefix_length)
print("Dropout         :", CONFIG.dropout)

ARCHITECTURE 4 — HYBRID PREFIX ENCODER LANGUAGE MODEL
Vocabulary size : 32004
Block size      : 2048
d_model         : 768
Layers          : 16
Heads           : 12
Head dimension  : 64
FFN dimension   : 2048
Prefix length   : 512
Dropout         : 0.0


In [20]:
# ============================================================
# CELL 10 — RMSNorm
# ============================================================

class RMSNorm(nn.Module):
    """
    Root Mean Square Layer Normalization.

    RMSNorm normalizes each token representation using its
    root-mean-square value, without subtracting the mean.
    """

    def __init__(self, dim, eps=1e-5):
        super().__init__()

        self.eps = eps

        # Learnable scale parameter
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        # x: [batch, sequence_length, d_model]

        # Compute RMS over the final dimension
        rms = x.pow(2).mean(dim=-1, keepdim=True)

        # Normalize
        x = x * torch.rsqrt(rms + self.eps)

        # Learnable scaling
        return self.weight * x

In [21]:
# ============================================================
# CELL 11 — SDPA BIDIRECTIONAL PREFIX ENCODER ATTENTION
# ============================================================

class PrefixEncoderAttention(nn.Module):
    """
    Bidirectional self-attention over the prefix only.

    Prefix tokens can attend to ALL other prefix tokens.

    Example:

        Prefix = [t0, t1, t2, t3]

        t0 <-> t1 <-> t2 <-> t3

    Tokens outside the prefix are not processed by this branch.
    """

    def __init__(
        self,
        d_model,
        n_heads,
        dropout=0.0,
        bias=False,
    ):
        super().__init__()

        assert d_model % n_heads == 0, (
            "d_model must be divisible by n_heads"
        )

        self.d_model = d_model
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads

        # ----------------------------------------------------
        # QKV projections
        # ----------------------------------------------------

        self.q_proj = nn.Linear(
            d_model,
            d_model,
            bias=bias,
        )

        self.k_proj = nn.Linear(
            d_model,
            d_model,
            bias=bias,
        )

        self.v_proj = nn.Linear(
            d_model,
            d_model,
            bias=bias,
        )

        # ----------------------------------------------------
        # Output projection
        # ----------------------------------------------------

        self.out_proj = nn.Linear(
            d_model,
            d_model,
            bias=bias,
        )

        self.dropout = nn.Dropout(
            dropout
        )


    def forward(
        self,
        x,
        prefix_length,
    ):
        """
        Args:
            x:
                [B, T, D]

            prefix_length:
                Number of tokens using bidirectional
                encoder attention.

        Returns:
            [B, T, D]
        """

        B, T, D = x.shape

        # ----------------------------------------------------
        # Clamp prefix length to current sequence length
        # ----------------------------------------------------

        prefix_length = min(
            prefix_length,
            T,
        )

        # ----------------------------------------------------
        # Output tensor.
        #
        # Only prefix positions will be filled.
        # Remaining positions stay zero.
        # ----------------------------------------------------

        encoder_output = torch.zeros_like(x)

        # ----------------------------------------------------
        # No prefix -> nothing to do
        # ----------------------------------------------------

        if prefix_length <= 0:
            return encoder_output

        # ----------------------------------------------------
        # Extract prefix
        # ----------------------------------------------------

        prefix = x[:, :prefix_length, :]

        # ----------------------------------------------------
        # Project Q, K, V
        # ----------------------------------------------------

        q = self.q_proj(prefix)
        k = self.k_proj(prefix)
        v = self.v_proj(prefix)

        # ----------------------------------------------------
        # Reshape:
        #
        # [B, P, D]
        #
        # ->
        #
        # [B, H, P, head_dim]
        # ----------------------------------------------------

        q = q.view(
            B,
            prefix_length,
            self.n_heads,
            self.head_dim,
        ).transpose(1, 2)

        k = k.view(
            B,
            prefix_length,
            self.n_heads,
            self.head_dim,
        ).transpose(1, 2)

        v = v.view(
            B,
            prefix_length,
            self.n_heads,
            self.head_dim,
        ).transpose(1, 2)

        # ----------------------------------------------------
        # SDPA
        #
        # is_causal=False
        #
        # Therefore each prefix token can attend to every
        # other prefix token.
        # ----------------------------------------------------

        attn_output = F.scaled_dot_product_attention(
            q,
            k,
            v,
            attn_mask=None,
            dropout_p=(
                self.dropout.p
                if self.training
                else 0.0
            ),
            is_causal=False,
        )

        # ----------------------------------------------------
        # [B, H, P, head_dim]
        #
        # ->
        #
        # [B, P, D]
        # ----------------------------------------------------

        attn_output = (
            attn_output
            .transpose(1, 2)
            .contiguous()
            .view(
                B,
                prefix_length,
                D,
            )
        )

        # ----------------------------------------------------
        # Output projection
        # ----------------------------------------------------

        attn_output = self.out_proj(
            attn_output
        )

        # ----------------------------------------------------
        # Put prefix output back into full sequence
        # ----------------------------------------------------

        encoder_output[
            :, :prefix_length, :
        ] = attn_output

        return encoder_output

In [22]:
# ============================================================
# CELL 12 — SDPA CAUSAL SELF-ATTENTION
# ============================================================

class CausalAttention(nn.Module):
    """
    Causal self-attention for autoregressive language modeling.

    Token at position t can attend only to:
        0 ... t

    It cannot attend to:
        t+1 ... T-1
    """

    def __init__(
        self,
        d_model,
        n_heads,
        block_size,
        dropout=0.0,
        bias=False,
    ):
        super().__init__()

        assert d_model % n_heads == 0, (
            "d_model must be divisible by n_heads"
        )

        self.d_model = d_model
        self.n_heads = n_heads
        self.head_dim = (
            d_model // n_heads
        )

        # ----------------------------------------------------
        # Q, K, V projections
        # ----------------------------------------------------

        self.q_proj = nn.Linear(
            d_model,
            d_model,
            bias=bias,
        )

        self.k_proj = nn.Linear(
            d_model,
            d_model,
            bias=bias,
        )

        self.v_proj = nn.Linear(
            d_model,
            d_model,
            bias=bias,
        )

        # ----------------------------------------------------
        # Output projection
        # ----------------------------------------------------

        self.out_proj = nn.Linear(
            d_model,
            d_model,
            bias=bias,
        )

        self.dropout = nn.Dropout(
            dropout
        )

        # block_size is kept in the constructor
        # for consistency with the rest of the model.
        self.block_size = block_size


    def forward(self, x):
        """
        Args:
            x:
                [B, T, D]

        Returns:
            [B, T, D]
        """

        B, T, D = x.shape

        # ----------------------------------------------------
        # Safety check
        # ----------------------------------------------------

        if T > self.block_size:

            raise ValueError(
                f"Sequence length {T} exceeds "
                f"block size {self.block_size}"
            )

        # ----------------------------------------------------
        # Q, K, V projections
        # ----------------------------------------------------

        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        # ----------------------------------------------------
        # Reshape:
        #
        # [B, T, D]
        #
        # ->
        #
        # [B, H, T, head_dim]
        # ----------------------------------------------------

        q = q.view(
            B,
            T,
            self.n_heads,
            self.head_dim,
        ).transpose(1, 2)

        k = k.view(
            B,
            T,
            self.n_heads,
            self.head_dim,
        ).transpose(1, 2)

        v = v.view(
            B,
            T,
            self.n_heads,
            self.head_dim,
        ).transpose(1, 2)

        # ----------------------------------------------------
        # Scaled Dot-Product Attention
        #
        # IMPORTANT:
        #
        # is_causal=True
        #
        # This automatically prevents future-token access.
        # ----------------------------------------------------

        attn_output = F.scaled_dot_product_attention(
            q,
            k,
            v,
            attn_mask=None,
            dropout_p=(
                self.dropout.p
                if self.training
                else 0.0
            ),
            is_causal=True,
        )

        # ----------------------------------------------------
        # [B, H, T, head_dim]
        #
        # ->
        #
        # [B, T, D]
        # ----------------------------------------------------

        attn_output = (
            attn_output
            .transpose(1, 2)
            .contiguous()
            .view(
                B,
                T,
                D,
            )
        )

        # ----------------------------------------------------
        # Output projection
        # ----------------------------------------------------

        attn_output = self.out_proj(
            attn_output
        )

        return attn_output

In [23]:
# ============================================================
# CELL 13 — GATED HYBRID FUSION
# ============================================================

class GatedHybridFusion(nn.Module):
    """
    Learned fusion of:

        1. Encoder-style representation
        2. Causal representation

    The model learns a gate:

        g = sigmoid(W [H_encoder ; H_causal])

    and combines them as:

        H = g * H_encoder
            + (1 - g) * H_causal
    """

    def __init__(
        self,
        d_model,
        bias=False,
    ):
        super().__init__()

        # ----------------------------------------------------
        # Gate projection
        #
        # Input:
        #   [H_encoder ; H_causal]
        #
        # Dimension:
        #   2 * d_model
        #
        # Output:
        #   d_model gates
        # ----------------------------------------------------

        self.gate_proj = nn.Linear(
            2 * d_model,
            d_model,
            bias=bias,
        )

        # ----------------------------------------------------
        # Final projection
        #
        # Allows the fused representation to be transformed
        # before being passed to the next component.
        # ----------------------------------------------------

        self.out_proj = nn.Linear(
            d_model,
            d_model,
            bias=bias,
        )


    def forward(
        self,
        encoder_output,
        causal_output,
    ):
        """
        encoder_output:
            [B, T, D]

        causal_output:
            [B, T, D]

        returns:
            fused_output:
                [B, T, D]

            gate:
                [B, T, D]
        """

        # ----------------------------------------------------
        # Concatenate the two representations
        # ----------------------------------------------------

        combined = torch.cat(
            [
                encoder_output,
                causal_output,
            ],
            dim=-1,
        )

        # ----------------------------------------------------
        # Learn a gate between 0 and 1
        # ----------------------------------------------------

        gate = torch.sigmoid(
            self.gate_proj(combined)
        )

        # ----------------------------------------------------
        # Hybrid representation
        #
        # gate close to 1:
        #     more encoder information
        #
        # gate close to 0:
        #     more causal information
        # ----------------------------------------------------

        fused = (
            gate * encoder_output
            + (1.0 - gate) * causal_output
        )

        # ----------------------------------------------------
        # Final projection
        # ----------------------------------------------------

        fused = self.out_proj(
            fused
        )

        return fused, gate

In [24]:
# ============================================================
# CELL 14 — SwiGLU FEED-FORWARD NETWORK
# ============================================================

class SwiGLU(nn.Module):
    """
    SwiGLU Feed-Forward Network.

    Formula:

        gate = SiLU(W_gate x)
        up   = W_up x

        hidden = gate * up

        output = W_down hidden

    Input:
        [B, T, d_model]

    Output:
        [B, T, d_model]
    """

    def __init__(
        self,
        d_model,
        ffn_dim,
        dropout=0.0,
        bias=False,
    ):
        super().__init__()

        # ----------------------------------------------------
        # Gate projection
        # ----------------------------------------------------

        self.gate_proj = nn.Linear(
            d_model,
            ffn_dim,
            bias=bias,
        )

        # ----------------------------------------------------
        # Up projection
        # ----------------------------------------------------

        self.up_proj = nn.Linear(
            d_model,
            ffn_dim,
            bias=bias,
        )

        # ----------------------------------------------------
        # Down projection
        # ----------------------------------------------------

        self.down_proj = nn.Linear(
            ffn_dim,
            d_model,
            bias=bias,
        )

        self.dropout = nn.Dropout(
            dropout
        )


    def forward(self, x):
        # ----------------------------------------------------
        # Gated branch
        # ----------------------------------------------------

        gate = F.silu(
            self.gate_proj(x)
        )

        # ----------------------------------------------------
        # Value branch
        # ----------------------------------------------------

        up = self.up_proj(x)

        # ----------------------------------------------------
        # SwiGLU interaction
        # ----------------------------------------------------

        hidden = gate * up

        # ----------------------------------------------------
        # Project back to d_model
        # ----------------------------------------------------

        output = self.down_proj(
            hidden
        )

        output = self.dropout(
            output
        )

        return output

In [25]:
# ============================================================
# CELL 15 — HYBRID TRANSFORMER BLOCK
# ============================================================

class HybridTransformerBlock(nn.Module):
    """
    One Hybrid Encoder/Language-Modeling Transformer block.

    Structure:

        X
        │
        ├── RMSNorm → Prefix Encoder Attention ──┐
        │                                        │
        └── RMSNorm → Causal Attention ──────────┤
                                                 │
                                          Gated Fusion
                                                 │
                                                 ▼
                                           Residual Add
                                                 │
                                                 ▼
                                              RMSNorm
                                                 │
                                                 ▼
                                               SwiGLU
                                                 │
                                                 ▼
                                           Residual Add
                                                 │
                                                 ▼
                                                Output
    """

    def __init__(self, config):
        super().__init__()

        # ----------------------------------------------------
        # Normalization before attention
        # ----------------------------------------------------

        self.encoder_norm = RMSNorm(
            config.d_model,
            config.norm_eps,
        )

        self.causal_norm = RMSNorm(
            config.d_model,
            config.norm_eps,
        )

        # ----------------------------------------------------
        # Encoder-style attention
        # ----------------------------------------------------

        self.encoder_attention = PrefixEncoderAttention(
            d_model=config.d_model,
            n_heads=config.n_heads,
            dropout=config.dropout,
            bias=config.bias,
        )

        # ----------------------------------------------------
        # Causal attention
        # ----------------------------------------------------

        self.causal_attention = CausalAttention(
            d_model=config.d_model,
            n_heads=config.n_heads,
            block_size=config.block_size,
            dropout=config.dropout,
            bias=config.bias,
        )

        # ----------------------------------------------------
        # Learned hybrid fusion
        # ----------------------------------------------------

        self.hybrid_fusion = GatedHybridFusion(
            d_model=config.d_model,
            bias=config.bias,
        )

        # ----------------------------------------------------
        # Normalization before FFN
        # ----------------------------------------------------

        self.ffn_norm = RMSNorm(
            config.d_model,
            config.norm_eps,
        )

        # ----------------------------------------------------
        # SwiGLU FFN
        # ----------------------------------------------------

        self.ffn = SwiGLU(
            d_model=config.d_model,
            ffn_dim=config.ffn_dim,
            dropout=config.dropout,
            bias=config.bias,
        )

    def forward(
        self,
        x,
        prefix_length,
        return_gate=False,
    ):
        """
        x:
            [B, T, D]

        prefix_length:
            Number of tokens handled by the
            bidirectional encoder branch.

        return_gate:
            If True, also return the learned
            fusion gate for inspection.
        """

        # ====================================================
        # ATTENTION BRANCHES
        # ====================================================

        # ----------------------------------------------------
        # Encoder branch
        # ----------------------------------------------------

        encoder_input = self.encoder_norm(x)

        encoder_output = self.encoder_attention(
            encoder_input,
            prefix_length,
        )

        # ----------------------------------------------------
        # Causal branch
        # ----------------------------------------------------

        causal_input = self.causal_norm(x)

        causal_output = self.causal_attention(
            causal_input
        )

        # ====================================================
        # HYBRID FUSION
        # ====================================================

        fused_output, gate = self.hybrid_fusion(
            encoder_output,
            causal_output,
        )

        # ----------------------------------------------------
        # Attention residual connection
        # ----------------------------------------------------

        x = x + fused_output

        # ====================================================
        # FEED-FORWARD NETWORK
        # ====================================================

        ffn_input = self.ffn_norm(x)

        ffn_output = self.ffn(
            ffn_input
        )

        # ----------------------------------------------------
        # FFN residual connection
        # ----------------------------------------------------

        x = x + ffn_output

        # ====================================================
        # OUTPUT
        # ====================================================

        if return_gate:
            return x, gate

        return x

In [26]:
# ============================================================
# CELL 16 — COMPLETE HYBRID LANGUAGE MODEL
# ============================================================

class HybridLM(nn.Module):
    """
    Architecture 4:
    Hybrid Prefix Encoder Language Model

    Combines:

        - Bidirectional encoder-style attention
          over a prefix

        - Causal self-attention
          over the complete sequence

        - Learned gated fusion

        - SwiGLU feed-forward network

    The model remains a single-stream language model.
    """

    def __init__(self, config):
        super().__init__()

        self.config = config

        # ----------------------------------------------------
        # Token embedding
        # ----------------------------------------------------

        self.token_embedding = nn.Embedding(
            config.vocab_size,
            config.d_model,
        )

        # ----------------------------------------------------
        # Learned positional embedding
        # ----------------------------------------------------

        self.position_embedding = nn.Embedding(
            config.block_size,
            config.d_model,
        )

        # ----------------------------------------------------
        # Embedding dropout
        # ----------------------------------------------------

        self.embedding_dropout = nn.Dropout(
            config.dropout
        )

        # ----------------------------------------------------
        # Hybrid Transformer blocks
        # ----------------------------------------------------

        self.blocks = nn.ModuleList(
            [
                HybridTransformerBlock(config)
                for _ in range(config.n_layers)
            ]
        )

        # ----------------------------------------------------
        # Final normalization
        # ----------------------------------------------------

        self.final_norm = RMSNorm(
            config.d_model,
            config.norm_eps,
        )

        # ----------------------------------------------------
        # Language-model head
        # ----------------------------------------------------

        self.lm_head = nn.Linear(
            config.d_model,
            config.vocab_size,
            bias=False,
        )

        # ----------------------------------------------------
        # Weight tying
        # ----------------------------------------------------

        self.lm_head.weight = self.token_embedding.weight

        # ----------------------------------------------------
        # Initialize weights
        # ----------------------------------------------------

        self.apply(self._init_weights)

    # ========================================================
    # WEIGHT INITIALIZATION
    # ========================================================

    def _init_weights(self, module):
        """
        Initialize model parameters.
        """

        if isinstance(module, nn.Linear):

            nn.init.normal_(
                module.weight,
                mean=0.0,
                std=0.02,
            )

            if module.bias is not None:
                nn.init.zeros_(module.bias)

        elif isinstance(module, nn.Embedding):

            nn.init.normal_(
                module.weight,
                mean=0.0,
                std=0.02,
            )

    # ========================================================
    # FORWARD
    # ========================================================

    def forward(
        self,
        input_ids,
        prefix_length=None,
        targets=None,
        return_gates=False,
    ):
        """
        input_ids:
            [B, T]

        prefix_length:
            Number of initial tokens processed by
            bidirectional encoder attention.

        targets:
            [B, T]

            Optional next-token targets.

        return_gates:
            If True, return the learned gates from
            every hybrid block.
        """

        B, T = input_ids.shape

        # ----------------------------------------------------
        # Sequence length check
        # ----------------------------------------------------

        if T > self.config.block_size:
            raise ValueError(
                f"Sequence length {T} exceeds "
                f"block size {self.config.block_size}"
            )

        # ----------------------------------------------------
        # Prefix length
        # ----------------------------------------------------

        if prefix_length is None:
            prefix_length = self.config.prefix_length

        prefix_length = min(
            prefix_length,
            T,
        )

        # ----------------------------------------------------
        # Position IDs
        # ----------------------------------------------------

        position_ids = torch.arange(
            T,
            device=input_ids.device,
        )

        # ----------------------------------------------------
        # Token + positional embeddings
        # ----------------------------------------------------

        x = (
            self.token_embedding(input_ids)
            +
            self.position_embedding(position_ids)
        )

        x = self.embedding_dropout(x)

        # ----------------------------------------------------
        # Store gates if requested
        # ----------------------------------------------------

        all_gates = []

        # ----------------------------------------------------
        # Hybrid Transformer stack
        # ----------------------------------------------------

        for block in self.blocks:

            if return_gates:

                x, gate = block(
                    x,
                    prefix_length=prefix_length,
                    return_gate=True,
                )

                all_gates.append(gate)

            else:

                x = block(
                    x,
                    prefix_length=prefix_length,
                    return_gate=False,
                )

        # ----------------------------------------------------
        # Final normalization
        # ----------------------------------------------------

        x = self.final_norm(x)

        # ----------------------------------------------------
        # Language-model logits
        # ----------------------------------------------------

        logits = self.lm_head(x)

        # ----------------------------------------------------
        # Optional language-model loss
        # ----------------------------------------------------

        loss = None

        if targets is not None:

            # -----------------------------------------------
            # IMPORTANT:
            # Prevent target leakage from the
            # bidirectional prefix.
            #
            # For position i:
            #
            #   input  = token i
            #   target = token i+1
            #
            # For i < prefix_length - 1,
            # target token i+1 is still inside
            # the bidirectional prefix.
            #
            # Therefore ignore those positions.
            # -----------------------------------------------

            loss_targets = targets.clone()

            if prefix_length > 1:

                loss_targets[:, :prefix_length - 1] = -100

            # -----------------------------------------------
            # Cross entropy only uses positions whose
            # target is not -100.
            # -----------------------------------------------

            loss = F.cross_entropy(
                logits.reshape(-1, logits.size(-1)),
                loss_targets.reshape(-1),
                ignore_index=-100,
            )

        # ----------------------------------------------------
        # Return
        # ----------------------------------------------------

        if return_gates:

            return {
                "logits": logits,
                "loss": loss,
                "gates": all_gates,
            }

        return logits, loss

In [27]:
# ============================================================
# CELL 17 — MODEL INSTANTIATION + PARAMETER COUNT
# ============================================================

# ------------------------------------------------------------
# Device
# ------------------------------------------------------------

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("=" * 70)
print("MODEL DEVICE")
print("=" * 70)

print("Device:", device)

if device.type == "cuda":
    print("GPU   :", torch.cuda.get_device_name(0))
else:
    print("WARNING: CUDA is not available.")

print("=" * 70)


# ------------------------------------------------------------
# Create model
# ------------------------------------------------------------

model = HybridLM(
    CONFIG
).to(device)


# ------------------------------------------------------------
# Count unique parameters
# ------------------------------------------------------------

def count_parameters(model):
    """
    Count total and trainable parameters.

    Shared parameters are counted only once.
    """

    seen = set()

    total_params = 0
    trainable_params = 0

    for parameter in model.parameters():

        parameter_id = id(parameter)

        if parameter_id in seen:
            continue

        seen.add(parameter_id)

        total_params += parameter.numel()

        if parameter.requires_grad:
            trainable_params += parameter.numel()

    return total_params, trainable_params


total_params, trainable_params = count_parameters(
    model
)


# ------------------------------------------------------------
# Display total parameters
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("HYBRIDLM PARAMETER COUNT")
print("=" * 70)

print(
    f"Total parameters       : {total_params:,}"
)

print(
    f"Trainable parameters   : {trainable_params:,}"
)

print(
    f"Total parameters (M)   : "
    f"{total_params / 1e6:.2f} M"
)

print(
    f"Trainable parameters(M): "
    f"{trainable_params / 1e6:.2f} M"
)

print("=" * 70)


# ------------------------------------------------------------
# Parameter breakdown
# ------------------------------------------------------------

print("\nParameter breakdown:")

# Important:
# lm_head.weight and token_embedding.weight are tied.
# Therefore shared parameters must be counted only once.

seen = set()

for name, module in model.named_children():

    module_params = 0

    for parameter in module.parameters():

        parameter_id = id(parameter)

        if parameter_id in seen:
            continue

        seen.add(parameter_id)

        module_params += parameter.numel()

    print(
        f"{name:25s} : "
        f"{module_params:,} "
        f"({module_params / 1e6:.2f} M)"
    )

print("=" * 70)

MODEL DEVICE
Device: cuda
GPU   : Tesla T4

HYBRIDLM PARAMETER COUNT
Total parameters       : 205,496,064
Trainable parameters   : 205,496,064
Total parameters (M)   : 205.50 M
Trainable parameters(M): 205.50 M

Parameter breakdown:
token_embedding           : 24,579,072 (24.58 M)
position_embedding        : 1,572,864 (1.57 M)
embedding_dropout         : 0 (0.00 M)
blocks                    : 179,343,360 (179.34 M)
final_norm                : 768 (0.00 M)
lm_head                   : 0 (0.00 M)


In [28]:
# ============================================================
# CELL 18 — FORWARD PASS + SHAPE VERIFICATION
# ============================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

# ------------------------------------------------------------
# Move HybridLM to device
# ------------------------------------------------------------

model = model.to(device)


# ------------------------------------------------------------
# Test configuration
# ------------------------------------------------------------

BATCH_SIZE_TEST = 2
SEQ_LEN_TEST = 256

# Use a smaller prefix so that this test exercises
# both the bidirectional prefix branch and the
# causal suffix branch.

PREFIX_LEN_TEST = 64


# ------------------------------------------------------------
# Random token IDs
# ------------------------------------------------------------

idx = torch.randint(
    low=0,
    high=VOCAB_SIZE,
    size=(
        BATCH_SIZE_TEST,
        SEQ_LEN_TEST,
    ),
    device=device,
)


# ------------------------------------------------------------
# Random next-token targets
# ------------------------------------------------------------

targets = torch.randint(
    low=0,
    high=VOCAB_SIZE,
    size=(
        BATCH_SIZE_TEST,
        SEQ_LEN_TEST,
    ),
    device=device,
)


# ------------------------------------------------------------
# Forward pass
# ------------------------------------------------------------

model.eval()

with torch.no_grad():

    logits, loss = model(
        idx,
        prefix_length=PREFIX_LEN_TEST,
        targets=targets,
    )


# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("HYBRIDLM FORWARD PASS CHECK")
print("=" * 70)

print(
    "Device        :",
    device
)

print(
    "Input shape   :",
    tuple(idx.shape)
)

print(
    "Prefix length :",
    PREFIX_LEN_TEST
)

print(
    "Logits shape  :",
    tuple(logits.shape)
)

print(
    "Loss          :",
    loss.item()
)


# ------------------------------------------------------------
# Assertions
# ------------------------------------------------------------

assert idx.shape == (
    BATCH_SIZE_TEST,
    SEQ_LEN_TEST,
)

assert targets.shape == (
    BATCH_SIZE_TEST,
    SEQ_LEN_TEST,
)

assert logits.shape == (
    BATCH_SIZE_TEST,
    SEQ_LEN_TEST,
    VOCAB_SIZE,
)

assert loss.ndim == 0


print("\n✓ Input shape verified")
print("✓ Target shape verified")
print("✓ Prefix length verified")
print("✓ Logits shape verified")
print("✓ Loss computation verified")
print("✓ HybridLM forward pass successful")

Device: cuda

HYBRIDLM FORWARD PASS CHECK
Device        : cuda
Input shape   : (2, 256)
Prefix length : 64
Logits shape  : (2, 256, 32004)
Loss          : 10.522368431091309

✓ Input shape verified
✓ Target shape verified
✓ Prefix length verified
✓ Logits shape verified
✓ Loss computation verified
✓ HybridLM forward pass successful


In [29]:
# ============================================================
# CELL 19 — CAUSAL / FUTURE-TOKEN LEAKAGE TEST
# ============================================================

model.eval()

# ------------------------------------------------------------
# Controlled test configuration
# ------------------------------------------------------------

B = 1
T = 32

# IMPORTANT:
# Keep prefix smaller than the checked position.
#
# Prefix:
#   0 ... 7
#
# Causal suffix:
#   8 ... 31
#
PREFIX_LEN_TEST = 8

# This position belongs to the causal suffix.
check_position = 10

# This future position is strictly after check_position.
future_position = 20


# ------------------------------------------------------------
# Create two almost-identical sequences
# ------------------------------------------------------------

x1 = torch.randint(
    0,
    VOCAB_SIZE,
    (B, T),
    device=device,
)

x2 = x1.clone()


# ------------------------------------------------------------
# Change ONLY a future token
# ------------------------------------------------------------

x2[0, future_position] = (
    x2[0, future_position] + 1
) % VOCAB_SIZE


# ------------------------------------------------------------
# Forward passes
# ------------------------------------------------------------

with torch.no_grad():

    logits1, _ = model(
        x1,
        prefix_length=PREFIX_LEN_TEST,
    )

    logits2, _ = model(
        x2,
        prefix_length=PREFIX_LEN_TEST,
    )


# ------------------------------------------------------------
# Compare an earlier causal position
# ------------------------------------------------------------

difference = (
    logits1[0, check_position]
    -
    logits2[0, check_position]
).abs().max().item()


# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

print("=" * 70)
print("CAUSAL FUTURE-TOKEN LEAKAGE TEST")
print("=" * 70)

print(
    "Sequence length        :",
    T
)

print(
    "Prefix length          :",
    PREFIX_LEN_TEST
)

print(
    "Changed future position:",
    future_position
)

print(
    "Checked causal position:",
    check_position
)

print(
    "Maximum logit difference:",
    difference
)


# ------------------------------------------------------------
# Verification
# ------------------------------------------------------------

if difference < 1e-5:

    print(
        "\n✓ No future-token leakage detected "
        "in the causal suffix."
    )

else:

    print(
        "\n⚠ Future information affected "
        "an earlier causal position."
    )

    print(
        "Difference:",
        difference
    )

CAUSAL FUTURE-TOKEN LEAKAGE TEST
Sequence length        : 32
Prefix length          : 8
Changed future position: 20
Checked causal position: 10
Maximum logit difference: 0.0

✓ No future-token leakage detected in the causal suffix.


In [30]:
# ============================================================
# CELL 20 — SINGLE TRAINING STEP
# ============================================================

import torch
import torch.nn as nn

# ------------------------------------------------------------
# Make sure model is on the selected device
# ------------------------------------------------------------

model = model.to(device)

# ------------------------------------------------------------
# Training mode
# ------------------------------------------------------------

model.train()

# ------------------------------------------------------------
# Small test batch
# ------------------------------------------------------------

BATCH_SIZE_TEST = 2
SEQ_LEN_TEST = 256

# Use a smaller prefix so that the test exercises
# BOTH the bidirectional prefix and causal suffix.

PREFIX_LEN_TEST = 64

idx = torch.randint(
    low=0,
    high=VOCAB_SIZE,
    size=(
        BATCH_SIZE_TEST,
        SEQ_LEN_TEST,
    ),
    device=device,
)

targets = torch.randint(
    low=0,
    high=VOCAB_SIZE,
    size=(
        BATCH_SIZE_TEST,
        SEQ_LEN_TEST,
    ),
    device=device,
)

# ------------------------------------------------------------
# Optimizer
# ------------------------------------------------------------

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=0.01,
)

# ------------------------------------------------------------
# Save one parameter before update
#
# This lets us verify that optimizer.step()
# actually changed the model.
# ------------------------------------------------------------

tracked_parameter = None

for p in model.parameters():
    if p.requires_grad:
        tracked_parameter = p.detach().clone()
        break

assert tracked_parameter is not None

# ------------------------------------------------------------
# Clear old gradients
# ------------------------------------------------------------

optimizer.zero_grad(
    set_to_none=True
)

# ------------------------------------------------------------
# Forward
# ------------------------------------------------------------

logits, loss = model(
    idx,
    prefix_length=PREFIX_LEN_TEST,
    targets=targets,
)

print("=" * 70)
print("SINGLE TRAINING STEP")
print("=" * 70)

print(
    "Input shape   :",
    tuple(idx.shape)
)

print(
    "Prefix length :",
    PREFIX_LEN_TEST
)

print(
    "Initial loss  :",
    loss.item()
)

# ------------------------------------------------------------
# Basic loss sanity check
# ------------------------------------------------------------

assert torch.isfinite(loss).item()
assert loss.ndim == 0

# ------------------------------------------------------------
# Backward
# ------------------------------------------------------------

loss.backward()

print("✓ Backward pass completed")

# ------------------------------------------------------------
# Gradient check
# ------------------------------------------------------------

total_grad_norm_sq = 0.0
num_params_with_grad = 0

for p in model.parameters():

    if p.grad is not None:

        grad_norm = p.grad.detach().norm(2).item()

        total_grad_norm_sq += (
            grad_norm ** 2
        )

        num_params_with_grad += 1

total_grad_norm = (
    total_grad_norm_sq ** 0.5
)

print(
    "Gradient norm:",
    total_grad_norm
)

print(
    "Parameters with gradients:",
    num_params_with_grad
)

assert num_params_with_grad > 0
assert total_grad_norm > 0
assert torch.isfinite(
    torch.tensor(total_grad_norm)
).item()

print("✓ Gradients are flowing")

# ------------------------------------------------------------
# Optimizer update
# ------------------------------------------------------------

optimizer.step()

print("✓ Optimizer step completed")

# ------------------------------------------------------------
# Verify that parameters changed
# ------------------------------------------------------------

parameter_changed = False

for p in model.parameters():

    if p.requires_grad:

        current_parameter = (
            p.detach()
        )

        if not torch.equal(
            current_parameter,
            tracked_parameter
        ):
            parameter_changed = True

        break

assert parameter_changed

print("✓ Model parameters were updated")

# ------------------------------------------------------------
# Second forward pass
# ------------------------------------------------------------

model.eval()

with torch.no_grad():

    _, new_loss = model(
        idx,
        prefix_length=PREFIX_LEN_TEST,
        targets=targets,
    )

print(
    "Loss after update:",
    new_loss.item()
)

assert torch.isfinite(
    new_loss
).item()

print("=" * 70)
print("✓ SINGLE TRAINING STEP PASSED")
print("=" * 70)

SINGLE TRAINING STEP
Input shape   : (2, 256)
Prefix length : 64
Initial loss  : 10.525083541870117
✓ Backward pass completed
Gradient norm: 12.26568964055558
Parameters with gradients: 259
✓ Gradients are flowing
✓ Optimizer step completed
✓ Model parameters were updated
Loss after update: 7.152353763580322
✓ SINGLE TRAINING STEP PASSED


In [31]:
# ============================================================
# CELL 21 — CREATE DATALOADERS FROM EXISTING DATASET
# ============================================================

from torch.utils.data import DataLoader

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

TRAIN_BATCH_SIZE = 1
NUM_WORKERS = 2

# ------------------------------------------------------------
# Use the dataset variables already created earlier
# ------------------------------------------------------------

train_dataset = CausalLMDataset(
    TRAIN_BIN,
    TOTAL_TRAIN_TOKENS,
    block_size=BLOCK_SIZE,
)

test_dataset = CausalLMDataset(
    TEST_BIN,
    TOTAL_TEST_TOKENS,
    block_size=BLOCK_SIZE,
)

print("=" * 70)
print("DATASET CHECK")
print("=" * 70)

print(
    "Train tokens :",
    f"{TOTAL_TRAIN_TOKENS:,}"
)

print(
    "Test tokens  :",
    f"{TOTAL_TEST_TOKENS:,}"
)

print(
    "Train samples:",
    f"{len(train_dataset):,}"
)

print(
    "Test samples :",
    f"{len(test_dataset):,}"
)

# ------------------------------------------------------------
# DataLoaders
# ------------------------------------------------------------

train_loader = DataLoader(
    train_dataset,
    batch_size=TRAIN_BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    drop_last=True,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=TRAIN_BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    drop_last=False,
)

print(
    "\nTrain batches:",
    f"{len(train_loader):,}"
)

print(
    "Test batches :",
    f"{len(test_loader):,}"
)

# ------------------------------------------------------------
# Load one real batch
# ------------------------------------------------------------

x_batch, y_batch = next(
    iter(train_loader)
)

print("\n" + "=" * 70)
print("REAL LEGAL BATCH")
print("=" * 70)

print(
    "Input shape :",
    tuple(x_batch.shape)
)

print(
    "Target shape:",
    tuple(y_batch.shape)
)

print(
    "Input dtype :",
    x_batch.dtype
)

print(
    "Target dtype:",
    y_batch.dtype
)

print(
    "Input range :",
    x_batch.min().item(),
    "to",
    x_batch.max().item(),
)

print(
    "Target range:",
    y_batch.min().item(),
    "to",
    y_batch.max().item(),
)

# ------------------------------------------------------------
# Verification
# ------------------------------------------------------------

assert x_batch.shape == (
    TRAIN_BATCH_SIZE,
    BLOCK_SIZE,
)

assert y_batch.shape == (
    TRAIN_BATCH_SIZE,
    BLOCK_SIZE,
)

assert x_batch.dtype == torch.long
assert y_batch.dtype == torch.long

assert (
    x_batch.min().item() >= 0
)

assert (
    x_batch.max().item() < VOCAB_SIZE
)

assert (
    y_batch.min().item() >= 0
)

assert (
    y_batch.max().item() < VOCAB_SIZE
)

print(
    "\n✓ Existing dataset reused"
)

print(
    "✓ Real legal batch loaded"
)

print(
    "✓ Shape verified"
)

print(
    "✓ Token range verified"
)

DATASET CHECK
Train tokens : 196,000,000
Test tokens  : 4,000,000
Train samples: 95,703
Test samples : 1,953

Train batches: 95,703
Test batches : 1,953

REAL LEGAL BATCH
Input shape : (1, 2048)
Target shape: (1, 2048)
Input dtype : torch.int64
Target dtype: torch.int64
Input range : 4 to 31685
Target range: 4 to 31685

✓ Existing dataset reused
✓ Real legal batch loaded
✓ Shape verified
✓ Token range verified


In [32]:
# ============================================================
# CELL 22 — ONE REAL LEGAL TRAINING STEP
# ============================================================

import torch

# ------------------------------------------------------------
# Make sure model is on the selected device
# ------------------------------------------------------------

model = model.to(device)

# ------------------------------------------------------------
# Training mode
# ------------------------------------------------------------

model.train()

# ------------------------------------------------------------
# Use the architecture's configured prefix
# ------------------------------------------------------------

PREFIX_LENGTH = CONFIG.prefix_length

print("=" * 70)
print("REAL LEGAL TRAINING STEP")
print("=" * 70)

print(
    "Device        :",
    device
)

print(
    "Prefix length :",
    PREFIX_LENGTH
)

# ------------------------------------------------------------
# Optimizer
#
# NOTE:
# This optimizer is only for this one-step sanity test.
# The real training loop should create ONE optimizer
# and reuse it across all training steps.
# ------------------------------------------------------------

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=0.01,
)

# ------------------------------------------------------------
# Get one REAL legal-language batch
# ------------------------------------------------------------

x_batch, y_batch = next(
    iter(train_loader)
)

x_batch = x_batch.to(
    device,
    non_blocking=True,
)

y_batch = y_batch.to(
    device,
    non_blocking=True,
)

print(
    "\nInput shape :",
    tuple(x_batch.shape)
)

print(
    "Target shape:",
    tuple(y_batch.shape)
)

# ------------------------------------------------------------
# Clear gradients
# ------------------------------------------------------------

optimizer.zero_grad(
    set_to_none=True
)

# ------------------------------------------------------------
# Forward pass
# ------------------------------------------------------------

if (
    "AMP_ENABLED" in globals()
    and AMP_ENABLED
    and device.type == "cuda"
):

    with torch.autocast(
        device_type="cuda",
        dtype=AMP_DTYPE,
    ):

        logits, loss = model(
            x_batch,
            prefix_length=PREFIX_LENGTH,
            targets=y_batch,
        )

else:

    logits, loss = model(
        x_batch,
        prefix_length=PREFIX_LENGTH,
        targets=y_batch,
    )

print(
    "\nLoss before backward:",
    loss.item()
)

# ------------------------------------------------------------
# Loss sanity check
# ------------------------------------------------------------

assert loss.ndim == 0
assert torch.isfinite(loss).item()

print("✓ Loss is finite")

# ------------------------------------------------------------
# Backward pass
# ------------------------------------------------------------

loss.backward()

print("✓ Backward pass completed")

# ------------------------------------------------------------
# Calculate gradient norm
# ------------------------------------------------------------

total_grad_norm_sq = 0.0
num_params_with_grad = 0

for parameter in model.parameters():

    if parameter.grad is not None:

        grad_norm = (
            parameter.grad.detach()
            .norm(2)
            .item()
        )

        total_grad_norm_sq += (
            grad_norm ** 2
        )

        num_params_with_grad += 1


total_grad_norm = (
    total_grad_norm_sq ** 0.5
)

print(
    "Gradient norm:",
    total_grad_norm
)

print(
    "Parameters with gradients:",
    num_params_with_grad
)

assert num_params_with_grad > 0
assert total_grad_norm > 0

print("✓ Gradients are flowing")

# ------------------------------------------------------------
# Optimizer update
# ------------------------------------------------------------

optimizer.step()

print("✓ Optimizer step completed")

# ------------------------------------------------------------
# Second forward pass
# ------------------------------------------------------------

model.eval()

with torch.no_grad():

    if (
        "AMP_ENABLED" in globals()
        and AMP_ENABLED
        and device.type == "cuda"
    ):

        with torch.autocast(
            device_type="cuda",
            dtype=AMP_DTYPE,
        ):

            _, new_loss = model(
                x_batch,
                prefix_length=PREFIX_LENGTH,
                targets=y_batch,
            )

    else:

        _, new_loss = model(
            x_batch,
            prefix_length=PREFIX_LENGTH,
            targets=y_batch,
        )

print(
    "\nLoss after update:",
    new_loss.item()
)

assert torch.isfinite(
    new_loss
).item()

print(
    "=" * 70
)

print(
    "✓ REAL LEGAL TRAINING STEP PASSED"
)

print(
    "=" * 70
)

REAL LEGAL TRAINING STEP
Device        : cuda
Prefix length : 512

Input shape : (1, 2048)
Target shape: (1, 2048)

Loss before backward: 10.545609474182129
✓ Loss is finite
✓ Backward pass completed
Gradient norm: 17.839146119302914
Parameters with gradients: 259
✓ Gradients are flowing
✓ Optimizer step completed

Loss after update: 8.081084251403809
✓ REAL LEGAL TRAINING STEP PASSED


In [33]:
# ============================================================
# CELL 23 — TRAINING CONFIG + CHECKPOINT SYSTEM
# ============================================================

import os
import random
from pathlib import Path

import numpy as np
import torch


# ============================================================
# TRAINING CONFIGURATION
# ============================================================

MAX_STEPS = 30000

LEARNING_RATE = 3e-4
MIN_LR = 3e-5

WEIGHT_DECAY = 0.1

WARMUP_STEPS = 200

GRAD_CLIP = 1.0

# Accumulate gradients across multiple micro-batches
GRAD_ACCUM_STEPS = 4

# Evaluation / checkpoint frequency
EVAL_INTERVAL = 250
CHECKPOINT_INTERVAL = 250

# Number of validation batches
EVAL_BATCHES = 20

# ============================================================
# CHECKPOINT DIRECTORY
# ============================================================

CHECKPOINT_DIR = Path(
    "/kaggle/working/model4_checkpoints"
)

CHECKPOINT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

LATEST_CHECKPOINT = (
    CHECKPOINT_DIR / "latest.pt"
)

BEST_CHECKPOINT = (
    CHECKPOINT_DIR / "best.pt"
)

# ============================================================
# MIXED PRECISION
# ============================================================

AMP_ENABLED = (
    device.type == "cuda"
)

AMP_DTYPE = torch.float16

print("=" * 70)
print("TRAINING CONFIGURATION")
print("=" * 70)

print("Device           :", device)
print("Max steps       :", MAX_STEPS)
print("Learning rate    :", LEARNING_RATE)
print("Minimum LR       :", MIN_LR)
print("Weight decay     :", WEIGHT_DECAY)
print("Warmup steps     :", WARMUP_STEPS)
print("Gradient clip    :", GRAD_CLIP)
print("Grad accumulation:", GRAD_ACCUM_STEPS)
print("Eval interval    :", EVAL_INTERVAL)
print("Checkpoint every :", CHECKPOINT_INTERVAL)
print("AMP enabled      :", AMP_ENABLED)
print("Checkpoint dir   :", CHECKPOINT_DIR)


# ============================================================
# OPTIMIZER
# ============================================================

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    betas=(0.9, 0.95),
    weight_decay=WEIGHT_DECAY,
)


# ============================================================
# LEARNING-RATE SCHEDULER
# ============================================================

def get_lr(step):
    """
    Linear warmup followed by cosine decay.
    """

    if step < WARMUP_STEPS:

        return (
            LEARNING_RATE
            * (step + 1)
            / WARMUP_STEPS
        )

    if step >= MAX_STEPS:

        return MIN_LR

    decay_ratio = (
        (step - WARMUP_STEPS)
        /
        (MAX_STEPS - WARMUP_STEPS)
    )

    cosine_coeff = 0.5 * (
        1.0 + np.cos(
            np.pi * decay_ratio
        )
    )

    return (
        MIN_LR
        + cosine_coeff
        * (LEARNING_RATE - MIN_LR)
    )


def update_learning_rate(step):
    """
    Update optimizer learning rate for the current step.
    """

    lr = get_lr(step)

    for param_group in optimizer.param_groups:
        param_group["lr"] = lr

    return lr


# ============================================================
# AMP GRADIENT SCALER
# ============================================================

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=AMP_ENABLED,
)


# ============================================================
# CHECKPOINT SAVE
# ============================================================

def save_checkpoint(
    path,
    step,
    best_val_loss,
):
    """
    Save everything needed to resume training.
    """

    checkpoint = {

        # ----------------------------------------------------
        # Model
        # ----------------------------------------------------

        "model_state_dict":
            model.state_dict(),

        # ----------------------------------------------------
        # Optimizer
        # ----------------------------------------------------

        "optimizer_state_dict":
            optimizer.state_dict(),

        # ----------------------------------------------------
        # AMP scaler
        # ----------------------------------------------------

        "scaler_state_dict":
            scaler.state_dict(),

        # ----------------------------------------------------
        # Training state
        # ----------------------------------------------------

        "step":
            step,

        "best_val_loss":
            best_val_loss,

        # ----------------------------------------------------
        # Configuration
        # ----------------------------------------------------

        "vocab_size":
            VOCAB_SIZE,

        "block_size":
            BLOCK_SIZE,

        "prefix_length":
            CONFIG.prefix_length,

        "model_config":
            vars(CONFIG),

        # ----------------------------------------------------
        # Random states
        # ----------------------------------------------------

        "python_rng_state":
            random.getstate(),

        "numpy_rng_state":
            np.random.get_state(),

        "torch_rng_state":
            torch.get_rng_state(),
    }

    if torch.cuda.is_available():

        checkpoint[
            "cuda_rng_state"
        ] = torch.cuda.get_rng_state_all()

    torch.save(
        checkpoint,
        path,
    )

    print(
        f"✓ Checkpoint saved: {path}"
    )


# ============================================================
# CHECKPOINT LOAD / RESUME
# ============================================================

def load_checkpoint(path):

    checkpoint = torch.load(
        path,
        map_location="cpu",
        weights_only=False,
    )

    # ========================================================
    # CONFIGURATION CHECKS
    # ========================================================

    checkpoint_vocab = checkpoint.get(
        "vocab_size",
        VOCAB_SIZE,
    )

    checkpoint_block = checkpoint.get(
        "block_size",
        BLOCK_SIZE,
    )

    checkpoint_prefix = checkpoint.get(
        "prefix_length",
        CONFIG.prefix_length,
    )

    assert checkpoint_vocab == VOCAB_SIZE, (
        f"Vocabulary mismatch: "
        f"checkpoint={checkpoint_vocab}, "
        f"current={VOCAB_SIZE}"
    )

    assert checkpoint_block == BLOCK_SIZE, (
        f"Block size mismatch: "
        f"checkpoint={checkpoint_block}, "
        f"current={BLOCK_SIZE}"
    )

    assert checkpoint_prefix == CONFIG.prefix_length, (
        f"Prefix mismatch: "
        f"checkpoint={checkpoint_prefix}, "
        f"current={CONFIG.prefix_length}"
    )

    # ========================================================
    # MODEL
    # ========================================================

    model.load_state_dict(
        checkpoint["model_state_dict"]
    )

    model.to(device)

    # ========================================================
    # OPTIMIZER
    # ========================================================

    optimizer.load_state_dict(
        checkpoint["optimizer_state_dict"]
    )

    # ========================================================
    # AMP SCALER
    # ========================================================

    if "scaler_state_dict" in checkpoint:

        scaler.load_state_dict(
            checkpoint[
                "scaler_state_dict"
            ]
        )

    # ========================================================
    # PYTHON RNG
    # ========================================================

    if "python_rng_state" in checkpoint:

        random.setstate(
            checkpoint[
                "python_rng_state"
            ]
        )

    # ========================================================
    # NUMPY RNG
    # ========================================================

    if "numpy_rng_state" in checkpoint:

        np.random.set_state(
            checkpoint[
                "numpy_rng_state"
            ]
        )

    # ========================================================
    # PYTORCH CPU RNG
    # ========================================================

    if "torch_rng_state" in checkpoint:

        state = checkpoint[
            "torch_rng_state"
        ]

        if not isinstance(
            state,
            torch.Tensor,
        ):

            state = torch.tensor(
                state,
                dtype=torch.uint8,
            )

        else:

            state = (
                state
                .detach()
                .cpu()
                .to(torch.uint8)
            )

        torch.set_rng_state(
            state
        )

    # ========================================================
    # CUDA RNG
    # ========================================================

    if (
        torch.cuda.is_available()
        and "cuda_rng_state" in checkpoint
    ):

        states = checkpoint[
            "cuda_rng_state"
        ]

        normalized_states = []

        for state in states:

            if not isinstance(
                state,
                torch.Tensor,
            ):

                state = torch.tensor(
                    state,
                    dtype=torch.uint8,
                )

            else:

                state = (
                    state
                    .detach()
                    .cpu()
                    .to(torch.uint8)
                )

            normalized_states.append(
                state
            )

        torch.cuda.set_rng_state_all(
            normalized_states
        )

    # ========================================================
    # TRAINING STATE
    # ========================================================

    step = checkpoint.get(
        "step",
        0,
    )

    best_val_loss = checkpoint.get(
        "best_val_loss",
        float("inf"),
    )

    print("=" * 70)
    print("CHECKPOINT RESUMED")
    print("=" * 70)

    print(
        "Checkpoint    :",
        path,
    )

    print(
        "Resume step   :",
        step,
    )

    print(
        "Best val loss :",
        best_val_loss,
    )

    return step, best_val_loss

# ============================================================
# RESUME SETTINGS
# ============================================================

RESUME_TRAINING = (
    LATEST_CHECKPOINT.exists()
)

START_STEP = 0

BEST_VAL_LOSS = float(
    "inf"
)


if RESUME_TRAINING:

    START_STEP, BEST_VAL_LOSS = (
        load_checkpoint(
            LATEST_CHECKPOINT
        )
    )

else:

    print(
        "\nNo previous checkpoint found."
    )

    print(
        "Starting training from step 0."
    )


print("\n✓ Training/checkpoint system ready")

TRAINING CONFIGURATION
Device           : cuda
Max steps       : 30000
Learning rate    : 0.0003
Minimum LR       : 3e-05
Weight decay     : 0.1
Warmup steps     : 200
Gradient clip    : 1.0
Grad accumulation: 4
Eval interval    : 250
Checkpoint every : 250
AMP enabled      : True
Checkpoint dir   : /kaggle/working/model4_checkpoints
CHECKPOINT RESUMED
Checkpoint    : /kaggle/working/model4_checkpoints/latest.pt
Resume step   : 30000
Best val loss : 1.263071495294571

✓ Training/checkpoint system ready


In [34]:
# ============================================================
# CELL 24 — VALIDATION + TRAINING UTILITIES
# ============================================================

import math
import time
import torch


# ============================================================
# VALIDATION FUNCTION
# ============================================================

def evaluate_model(
    model,
    data_loader,
    device,
    max_batches=None,
):
    """
    Evaluate the model on validation/test data.

    Returns:
        average_loss
        perplexity
    """

    model.eval()

    total_loss = 0.0
    total_batches = 0

    with torch.no_grad():

        for batch_idx, (x, y) in enumerate(data_loader):

            if (
                max_batches is not None
                and batch_idx >= max_batches
            ):
                break

            x = x.to(
                device,
                non_blocking=True,
            )

            y = y.to(
                device,
                non_blocking=True,
            )

            # ----------------------------------------------
            # Mixed precision only on CUDA
            # ----------------------------------------------

            if AMP_ENABLED:

                with torch.autocast(
                    device_type="cuda",
                    dtype=AMP_DTYPE,
                ):
                    _, loss = model(
                        x,
                        targets=y,
                    )

            else:

                _, loss = model(
                    x,
                    targets=y,
                )

            total_loss += loss.item()
            total_batches += 1

    if total_batches == 0:
        raise RuntimeError(
            "Validation loader produced zero batches."
        )

    avg_loss = (
        total_loss / total_batches
    )

    try:
        perplexity = math.exp(avg_loss)
    except OverflowError:
        perplexity = float("inf")

    model.train()

    return avg_loss, perplexity


# ============================================================
# DATA ITERATOR
# ============================================================

def infinite_train_loader(loader):
    """
    Continuously yield batches from the training loader.
    """

    while True:

        for batch in loader:
            yield batch


# ============================================================
# TRAINING HISTORY
# ============================================================

training_history = {
    "step": [],
    "train_loss": [],
    "val_loss": [],
    "val_perplexity": [],
    "learning_rate": [],
}


# ============================================================
# INITIAL STATUS
# ============================================================

print("=" * 70)
print("TRAINING UTILITIES READY")
print("=" * 70)

print(
    "Starting step   :",
    START_STEP,
)

print(
    "Maximum steps   :",
    MAX_STEPS,
)

print(
    "Best val loss   :",
    BEST_VAL_LOSS,
)

print(
    "AMP enabled     :",
    AMP_ENABLED,
)

print(
    "Checkpoint dir  :",
    CHECKPOINT_DIR,
)

print("✓ Validation and training utilities ready")

TRAINING UTILITIES READY
Starting step   : 30000
Maximum steps   : 30000
Best val loss   : 1.263071495294571
AMP enabled     : True
Checkpoint dir  : /kaggle/working/model4_checkpoints
✓ Validation and training utilities ready


In [35]:
# ============================================================
# CELL 25 — CHECKPOINT-BASED FULL TRAINING
# ============================================================

import time
import torch


# ============================================================
# MODEL
# ============================================================

model = model.to(device)
model.train()

PREFIX_LENGTH = min(
    CONFIG.prefix_length,
    BLOCK_SIZE,
)


# ============================================================
# TRAINING ITERATOR
# ============================================================

train_iterator = infinite_train_loader(
    train_loader
)


# ============================================================
# TRAINING STATE
# ============================================================

global_step = START_STEP

best_val_loss = BEST_VAL_LOSS

optimizer.zero_grad(
    set_to_none=True
)


# ============================================================
# TRAINING START
# ============================================================

print("=" * 70)
print("STARTING HYBRIDLM TRAINING")
print("=" * 70)

print(
    "Starting step       :",
    global_step,
)

print(
    "Target step         :",
    MAX_STEPS,
)

print(
    "Prefix length       :",
    PREFIX_LENGTH,
)

print(
    "Gradient accumulation:",
    GRAD_ACCUM_STEPS,
)

print(
    "Device              :",
    device,
)

print(
    "AMP enabled         :",
    AMP_ENABLED,
)

print("=" * 70)


training_start_time = time.time()


# ============================================================
# MAIN LOOP
# ============================================================

while global_step < MAX_STEPS:

    step_start_time = time.time()

    # --------------------------------------------------------
    # Learning rate for this optimizer step
    # --------------------------------------------------------

    lr = update_learning_rate(
        global_step
    )

    # --------------------------------------------------------
    # Accumulate gradients
    # --------------------------------------------------------

    accumulated_loss = 0.0

    optimizer.zero_grad(
        set_to_none=True
    )

    for micro_step in range(
        GRAD_ACCUM_STEPS
    ):

        # ----------------------------------------------------
        # Get real legal-language batch
        # ----------------------------------------------------

        x_batch, y_batch = next(
            train_iterator
        )

        x_batch = x_batch.to(
            device,
            non_blocking=True,
        )

        y_batch = y_batch.to(
            device,
            non_blocking=True,
        )

        # ----------------------------------------------------
        # Forward
        # ----------------------------------------------------

        if (
            AMP_ENABLED
            and device.type == "cuda"
        ):

            with torch.autocast(
                device_type="cuda",
                dtype=AMP_DTYPE,
            ):

                _, loss = model(
                    x_batch,
                    prefix_length=PREFIX_LENGTH,
                    targets=y_batch,
                )

        else:

            _, loss = model(
                x_batch,
                prefix_length=PREFIX_LENGTH,
                targets=y_batch,
            )

        # ----------------------------------------------------
        # Loss sanity check
        # ----------------------------------------------------

        if not torch.isfinite(loss).item():

            raise FloatingPointError(
                f"Non-finite training loss at "
                f"step {global_step}, "
                f"micro-step {micro_step}."
            )

        # ----------------------------------------------------
        # Normalize loss for gradient accumulation
        # ----------------------------------------------------

        loss_for_backward = (
            loss / GRAD_ACCUM_STEPS
        )

        # ----------------------------------------------------
        # Backward
        # ----------------------------------------------------

        if AMP_ENABLED:

            scaler.scale(
                loss_for_backward
            ).backward()

        else:

            loss_for_backward.backward()

        accumulated_loss += (
            loss.item()
        )

    # ========================================================
    # UNSCALE + GRADIENT CLIPPING
    # ========================================================

    if AMP_ENABLED:

        scaler.unscale_(
            optimizer
        )

    grad_norm = torch.nn.utils.clip_grad_norm_(
        model.parameters(),
        GRAD_CLIP,
        error_if_nonfinite=False,
    )

    # --------------------------------------------------------
    # Gradient sanity check
    # --------------------------------------------------------

    if not torch.isfinite(
        torch.as_tensor(
            grad_norm,
            device=device,
        )
    ).item():

        print(
            f"\n⚠ Non-finite gradient detected "
            f"at step {global_step}"
        )
        bad_parameters = []

        for name, parameter in model.named_parameters():

            if parameter.grad is None:
                continue

            if not torch.isfinite(
                parameter.grad
            ).all():

                bad_parameters.append(name)
    
        print(
            "Problematic parameters:"
        )

        for name in bad_parameters[:20]:
            print("  ", name)
    
        if len(bad_parameters) > 20:
            print(
                f"  ... and "
                f"{len(bad_parameters) - 20} more"
            )

        # --------------------------------------------
        # Discard this bad gradient
        # --------------------------------------------

        optimizer.zero_grad(
            set_to_none=True
        )

        # --------------------------------------------
        # Let GradScaler reduce its scale
        # --------------------------------------------

        if AMP_ENABLED:
            scaler.update()

        print(
            "✓ Bad optimizer update skipped"
        )
        continue

    # ========================================================
    # OPTIMIZER STEP
    # ========================================================

    if AMP_ENABLED:

        scaler.step(
            optimizer
        )

        scaler.update()

    else:

        optimizer.step()

    # ========================================================
    # STEP COMPLETE
    # ========================================================

    global_step += 1

    avg_train_loss = (
        accumulated_loss
        / GRAD_ACCUM_STEPS
    )

    step_time = (
        time.time()
        - step_start_time
    )

    # ========================================================
    # LOGGING
    # ========================================================

    if (
        global_step == 1
        or global_step % 10 == 0
    ):

        print(
            f"Step {global_step:5d}/{MAX_STEPS} | "
            f"train_loss={avg_train_loss:.4f} | "
            f"lr={lr:.6e} | "
            f"grad_norm={float(grad_norm):.4f} | "
            f"time={step_time:.2f}s"
        )

    # ========================================================
    # VALIDATION
    # ========================================================

    if (
        global_step % EVAL_INTERVAL == 0
        or global_step == MAX_STEPS
    ):

        print(
            "\n"
            + "-" * 70
        )

        print(
            f"Validation at step {global_step}"
        )

        val_loss, val_perplexity = (
            evaluate_model(
                model,
                test_loader,
                device,
                max_batches=EVAL_BATCHES,
            )
        )

        print(
            f"Train loss      : {avg_train_loss:.4f}"
        )

        print(
            f"Val loss        : {val_loss:.4f}"
        )

        print(
            f"Val perplexity  : {val_perplexity:.4f}"
        )

        # ----------------------------------------------------
        # Save history
        # ----------------------------------------------------

        training_history[
            "step"
        ].append(
            global_step
        )

        training_history[
            "train_loss"
        ].append(
            avg_train_loss
        )

        training_history[
            "val_loss"
        ].append(
            val_loss
        )

        training_history[
            "val_perplexity"
        ].append(
            val_perplexity
        )

        training_history[
            "learning_rate"
        ].append(
            lr
        )

        # ----------------------------------------------------
        # Best checkpoint
        # ----------------------------------------------------

        if val_loss < best_val_loss:

            best_val_loss = val_loss

            save_checkpoint(
                BEST_CHECKPOINT,
                global_step,
                best_val_loss,
            )

            print(
                f"✓ New best model "
                f"(val_loss={best_val_loss:.4f})"
            )

        print(
            "-" * 70
            + "\n"
        )

    # ========================================================
    # LATEST CHECKPOINT
    # ========================================================

    if (
        global_step % CHECKPOINT_INTERVAL == 0
        or global_step == MAX_STEPS
    ):

        save_checkpoint(
            LATEST_CHECKPOINT,
            global_step,
            best_val_loss,
        )

        print(
            f"✓ Latest checkpoint saved "
            f"at step {global_step}"
        )


# ============================================================
# TRAINING FINISHED
# ============================================================

total_training_time = (
    time.time()
    - training_start_time
)

print(
    "\n"
    + "=" * 70
)

print(
    "TRAINING FINISHED"
)

print(
    "=" * 70
)

print(
    "Final step       :",
    global_step,
)

print(
    "Best val loss    :",
    best_val_loss,
)

print(
    "Training time    :",
    f"{total_training_time / 3600:.2f} hours",
)

print(
    "Latest checkpoint:",
    LATEST_CHECKPOINT,
)

print(
    "Best checkpoint  :",
    BEST_CHECKPOINT,
)

print("=" * 70)

STARTING HYBRIDLM TRAINING
Starting step       : 30000
Target step         : 30000
Prefix length       : 512
Gradient accumulation: 4
Device              : cuda
AMP enabled         : True

TRAINING FINISHED
Final step       : 30000
Best val loss    : 1.263071495294571
Training time    : 0.00 hours
Latest checkpoint: /kaggle/working/model4_checkpoints/latest.pt
Best checkpoint  : /kaggle/working/model4_checkpoints/best.pt


In [36]:
# ============================================================
# CELL 26 — LOAD BEST MODEL FOR EVALUATION
# ============================================================

import torch

print("=" * 70)
print("LOADING BEST CHECKPOINT")
print("=" * 70)

if not BEST_CHECKPOINT.exists():
    raise FileNotFoundError(
        f"Best checkpoint not found:\n{BEST_CHECKPOINT}"
    )

# ------------------------------------------------------------
# Load trusted checkpoint
# ------------------------------------------------------------

checkpoint = torch.load(
    BEST_CHECKPOINT,
    map_location="cpu",
    weights_only=False,
)

# ------------------------------------------------------------
# Load model weights
# ------------------------------------------------------------

model.load_state_dict(
    checkpoint["model_state_dict"]
)

model = model.to(device)
model.eval()

# ------------------------------------------------------------
# Metadata
# ------------------------------------------------------------

best_step = checkpoint.get(
    "step",
    "unknown"
)

best_val_loss = checkpoint.get(
    "best_val_loss",
    "unknown"
)

print(
    "Checkpoint   :",
    BEST_CHECKPOINT
)

print(
    "Best step    :",
    best_step
)

print(
    "Best val loss:",
    best_val_loss
)

print(
    "Device       :",
    device
)

print("\n✓ Best model loaded successfully")

LOADING BEST CHECKPOINT
Checkpoint   : /kaggle/working/model4_checkpoints/best.pt
Best step    : 30000
Best val loss: 1.263071495294571
Device       : cuda

✓ Best model loaded successfully


In [37]:
# Hugging Face dataset
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"

TEST_FILE = "Phase_1_Pretraining/Test/test.json"

OUTPUT_DIR = Path("/kaggle/working/phase1_data")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [38]:
import os
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()

HF_TOKEN = user_secrets.get_secret("HF_TOKEN")

os.environ["HF_TOKEN"] = HF_TOKEN

login(token=HF_TOKEN)

print("Hugging Face login successful.")

Hugging Face login successful.


In [39]:
from huggingface_hub import hf_hub_download

print(f"Downloading {TEST_FILE} from {HF_DATASET_REPO}...")

test_data_path = hf_hub_download(
    repo_id=HF_DATASET_REPO,
    filename=TEST_FILE,
    repo_type="dataset",
    token=HF_TOKEN,
)

print("Path:", test_data_path)

with open(test_data_path, "r", encoding="utf-8") as f:
    test_data = json.load(f)

print("Loaded test examples:", len(test_data))

first_key = next(iter(test_data))
print("First ID:", first_key)
print("First prompt:")
print(test_data[first_key]["text"][:500])


Path: /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_1_Pretraining/Test/test.json
Loaded test examples: 1000
First ID: 1
First prompt:


In [40]:
# ============================================================
# LOAD TRAINED BPE TOKENIZER
# ============================================================

import sys
from pathlib import Path


# ------------------------------------------------------------
# PATH OF TOKENIZER CREATED IN TOKENIZER NOTEBOOK
# ------------------------------------------------------------

TOKENIZER_DIR = Path(
    "/kaggle/input/notebooks/ritikguptatab/updated-bpe-tokenizer/legal_tokenizer_v3/tokenizer/final_tokenizer"
)


if not TOKENIZER_DIR.exists():

    raise FileNotFoundError(
        f"Tokenizer directory not found:\n"
        f"{TOKENIZER_DIR}\n\n"
        "Make sure the tokenizer notebook has been "
        "run and its final_tokenizer directory is "
        "available in this notebook."
    )


TOKENIZER_PY = (
    TOKENIZER_DIR / "tokenizer.py"
)


if not TOKENIZER_PY.exists():

    raise FileNotFoundError(
        f"tokenizer.py not found:\n"
        f"{TOKENIZER_PY}"
    )


# ------------------------------------------------------------
# IMPORT tokenizer.py
# ------------------------------------------------------------

sys.path.insert(
    0,
    str(TOKENIZER_DIR)
)

from tokenizer import load_tokenizer


# ------------------------------------------------------------
# LOAD TOKENIZER
# ------------------------------------------------------------

tokenizer, tokenizer_config = (
    load_tokenizer(
        TOKENIZER_DIR
    )
)


print("=" * 70)
print("TOKENIZER LOADED")
print("=" * 70)

print(
    "Tokenizer directory:",
    TOKENIZER_DIR
)

print(
    "Vocabulary size:",
    tokenizer.get_vocab_size()
)

print(
    "PAD ID:",
    tokenizer.pad_id
)

print(
    "UNK ID:",
    tokenizer.unk_id
)

print(
    "BOS ID:",
    tokenizer.bos_id
)

print(
    "EOS ID:",
    tokenizer.eos_id
)


# ------------------------------------------------------------
# PROMPT LENGTH CHECK
#
# Prompts shorter than CONFIG.prefix_length tokens are outside the
# training distribution (the whole context becomes bidirectional
# prefix). Report how many there are.
# ------------------------------------------------------------

prompt_token_lengths = {
    k: len(tokenizer.encode(v["text"]))
    for k, v in test_data.items()
}

lengths = sorted(prompt_token_lengths.values())

print()
print("Prompt tokens  min / median / max:",
      lengths[0], "/", lengths[len(lengths) // 2], "/", lengths[-1])

print("Prompts shorter than prefix_length:",
      sum(l < CONFIG.prefix_length for l in lengths))

print("Prompts longer than block_size   :",
      sum(l > BLOCK_SIZE for l in lengths))


TOKENIZER LOADED
Tokenizer directory: /kaggle/input/notebooks/ritikguptatab/updated-bpe-tokenizer/legal_tokenizer_v3/tokenizer/final_tokenizer
Vocabulary size: 32004
PAD ID: 0
UNK ID: 1
BOS ID: 2
EOS ID: 3

Prompt tokens  min / median / max: 1122 / 1248 / 1643
Prompts shorter than prefix_length: 0
Prompts longer than block_size   : 0


In [41]:
# ============================================================
# TEST GENERATION — SETTINGS
# ============================================================

import json
import time
from pathlib import Path


# ------------------------------------------------------------
# OUTPUT FILES
# ------------------------------------------------------------

PREDICTION_PROGRESS_PATH = Path(
    "/kaggle/working/prediction_progress.json"
)

FAILURE_LOG_PATH = Path(
    "/kaggle/working/prediction_failures.json"
)

FINAL_PREDICTION_PATH = Path(
    "/kaggle/working/prediction.json"
)


# ------------------------------------------------------------
# GENERATION SETTINGS
# ------------------------------------------------------------

GEN_MAX_NEW_TOKENS = 512

GEN_TEMPERATURE = 0.8
GEN_TOP_K = 50
GEN_TOP_P = 0.95

# Penalise tokens already present in the recent context.
# 1.0 = off. 1.1 - 1.3 is a sensible range.
GEN_REPETITION_PENALTY = 1.15
GEN_REPETITION_WINDOW = 256

# Retries only help for transient errors (e.g. CUDA OOM).
MAX_RETRIES = 2

SAVE_EVERY_EXAMPLES = 10

# Number of test examples decoded together by the fast generator.
# Each sequence needs ~100 MB of KV cache on a T4 (fp16), so 16 uses
# ~1.6 GB. Lower it if you hit CUDA out-of-memory.
GEN_BATCH_SIZE = 16


print("=" * 70)
print("GENERATION SETTINGS")
print("=" * 70)
print("Max new tokens     :", GEN_MAX_NEW_TOKENS)
print("Temperature        :", GEN_TEMPERATURE)
print("Top-k / Top-p      :", GEN_TOP_K, "/", GEN_TOP_P)
print("Repetition penalty :", GEN_REPETITION_PENALTY,
      f"(window {GEN_REPETITION_WINDOW})")
print("Max retries        :", MAX_RETRIES)
print("Save every         :", SAVE_EVERY_EXAMPLES, "examples")
print("Batch size (fast)  :", GEN_BATCH_SIZE)


GENERATION SETTINGS
Max new tokens     : 512
Temperature        : 0.8
Top-k / Top-p      : 50 / 0.95
Repetition penalty : 1.15 (window 256)
Max retries        : 2
Save every         : 10 examples
Batch size (fast)  : 16


In [42]:
# ============================================================
# CELL 34 — VERIFY GENERATION ENVIRONMENT
# ============================================================

import json
import torch

print("=" * 70)
print("GENERATION ENVIRONMENT CHECK")
print("=" * 70)

# ------------------------------------------------------------
# Test data
# ------------------------------------------------------------

assert isinstance(
    test_data,
    dict
)

assert len(test_data) > 0

print(
    "Test examples :",
    len(test_data)
)

# ------------------------------------------------------------
# Inspect first example
# ------------------------------------------------------------

first_id = next(
    iter(test_data)
)

first_example = test_data[first_id]

assert isinstance(
    first_example,
    dict
)

assert "text" in first_example

print(
    "First ID      :",
    first_id
)

print(
    "Prompt length  :",
    len(first_example["text"])
)

# ------------------------------------------------------------
# Model
# ------------------------------------------------------------

model = model.to(device)
model.eval()

print(
    "Device        :",
    device
)

# ------------------------------------------------------------
# Tokenizer
# ------------------------------------------------------------

assert tokenizer.get_vocab_size() == VOCAB_SIZE

print(
    "Vocabulary    :",
    tokenizer.get_vocab_size()
)

print(
    "EOS ID        :",
    tokenizer.eos_id
)

# ------------------------------------------------------------
# Check model parameters
# ------------------------------------------------------------

with torch.no_grad():

    sample_ids = tokenizer.encode(
        first_example["text"]
    )

assert len(sample_ids) > 0

assert all(
    0 <= int(token_id) < VOCAB_SIZE
    for token_id in sample_ids
)

print(
    "Prompt tokens  :",
    len(sample_ids)
)

print(
    "\n✓ Test data verified"
)

print(
    "✓ Model verified"
)

print(
    "✓ Tokenizer verified"
)

print("=" * 70)

GENERATION ENVIRONMENT CHECK
Test examples : 1000
First ID      : 1
Prompt length  : 2889
Device        : cuda
Vocabulary    : 32004
EOS ID        : 3
Prompt tokens  : 1261

✓ Test data verified
✓ Model verified
✓ Tokenizer verified


In [43]:
# ============================================================
# CELL 35 — HYBRIDLM AUTOREGRESSIVE GENERATION
# ============================================================

import torch


def generate_tokens(
    model,
    input_ids,
    max_new_tokens=512,
    temperature=0.8,
    top_k=50,
    top_p=0.95,
    repetition_penalty=1.0,
    repetition_window=256,
):
    """
    Generate tokens autoregressively using HybridLM.

    input_ids : torch.Tensor of shape [1, T] (the prompt)

    Returns
    -------
    torch.Tensor of shape [1, n_new]
        ONLY the newly generated tokens (the prompt is NOT included).
        If EOS is produced it is included as the last token; use
        decode_continuation() to strip it.
    """

    if input_ids.ndim != 2:
        raise ValueError(
            f"input_ids must have shape [B, T], got {tuple(input_ids.shape)}"
        )

    if input_ids.size(0) != 1:
        raise ValueError("generate_tokens supports batch size = 1 only.")

    if max_new_tokens <= 0:
        return input_ids[:, :0]

    was_training = model.training
    model.eval()

    prompt_length = input_ids.size(1)
    generated = input_ids.clone()

    eos_id = getattr(tokenizer, "eos_id", None)

    try:

        with torch.no_grad():

            for _ in range(max_new_tokens):

                # ------------------------------------------------
                # Keep context inside the model window
                # ------------------------------------------------

                idx_cond = generated[:, -BLOCK_SIZE:]

                # First prefix_length positions use bidirectional
                # attention, exactly as in training.
                prefix_length = min(
                    CONFIG.prefix_length,
                    idx_cond.size(1),
                )

                # ------------------------------------------------
                # Forward
                # ------------------------------------------------

                if AMP_ENABLED and device.type == "cuda":
                    with torch.autocast(device_type="cuda", dtype=AMP_DTYPE):
                        logits, _ = model(idx_cond, prefix_length=prefix_length)
                else:
                    logits, _ = model(idx_cond, prefix_length=prefix_length)

                # Last position predicts the next token.
                # Cast to fp32: autocast returns fp16 logits, which are
                # fragile for softmax / sort / cumsum.
                logits = logits[:, -1, :].float()

                # ------------------------------------------------
                # Repetition penalty (CTRL-style)
                # ------------------------------------------------

                if repetition_penalty is not None and repetition_penalty != 1.0:

                    recent = generated[0, -repetition_window:]
                    prev_tokens = torch.unique(recent)

                    selected = logits[0, prev_tokens]

                    logits[0, prev_tokens] = torch.where(
                        selected > 0,
                        selected / repetition_penalty,
                        selected * repetition_penalty,
                    )

                # ------------------------------------------------
                # Greedy
                # ------------------------------------------------

                if temperature is None or temperature <= 0:

                    next_token = torch.argmax(logits, dim=-1, keepdim=True)

                else:

                    logits = logits / temperature

                    # ---------------- top-k ----------------
                    if top_k is not None and top_k > 0:

                        k = min(int(top_k), logits.size(-1))

                        topk_values, topk_indices = torch.topk(logits, k=k, dim=-1)

                        filtered = torch.full_like(logits, float("-inf"))
                        filtered.scatter_(dim=-1, index=topk_indices, src=topk_values)
                        logits = filtered

                    # ---------------- top-p ----------------
                    if top_p is not None and 0.0 < top_p < 1.0:

                        sorted_logits, sorted_indices = torch.sort(
                            logits, descending=True, dim=-1
                        )

                        cumulative_probs = torch.cumsum(
                            torch.softmax(sorted_logits, dim=-1), dim=-1
                        )

                        sorted_remove = cumulative_probs > top_p

                        # keep the token that crosses the threshold
                        sorted_remove[:, 1:] = sorted_remove[:, :-1].clone()
                        sorted_remove[:, 0] = False

                        sorted_logits = sorted_logits.masked_fill(
                            sorted_remove, float("-inf")
                        )

                        logits = torch.full_like(logits, float("-inf"))
                        logits.scatter_(dim=-1, index=sorted_indices, src=sorted_logits)

                    probs = torch.softmax(logits, dim=-1)

                    if not torch.isfinite(probs).all():
                        raise RuntimeError(
                            "Non-finite probabilities encountered during generation."
                        )

                    next_token = torch.multinomial(probs, num_samples=1)

                # ------------------------------------------------
                # Append + EOS stop
                # ------------------------------------------------

                generated = torch.cat([generated, next_token], dim=1)

                if eos_id is not None and next_token.item() == eos_id:
                    break

    finally:

        if was_training:
            model.train()

    # Return ONLY the newly generated tokens
    return generated[:, prompt_length:]


def decode_continuation(new_ids):
    """
    Decode the output of generate_tokens() to text.

    - Accepts a [1, n] tensor, a 1-D tensor, or a list of ids.
    - Cuts everything from the first EOS onwards, so the literal
      "<eos>" string never appears in predictions.
    """

    if isinstance(new_ids, torch.Tensor):
        if new_ids.ndim == 2:
            new_ids = new_ids[0]
        new_ids = new_ids.tolist()

    eos_id = getattr(tokenizer, "eos_id", None)

    if eos_id is not None and eos_id in new_ids:
        new_ids = new_ids[: new_ids.index(eos_id)]

    return tokenizer.decode(new_ids)


print("✓ generate_tokens() and decode_continuation() ready")


✓ generate_tokens() and decode_continuation() ready


In [44]:
# ============================================================
# CELL 35b — FAST GENERATION: KV CACHE + BATCHED DECODING
#
# Same model, same weights, same maths as generate_tokens(), but:
#
#   1. KV cache: the prompt is processed ONCE. Each new token then
#      only runs the model on that single token, attending to the
#      cached keys/values, instead of re-running ~1500 tokens.
#
#   2. Batching: GEN_BATCH_SIZE test examples are decoded together.
#
# Why the cache is exact for HybridLM:
#   - The prompt is kept to at most BLOCK_SIZE - max_new_tokens
#     tokens, so the context window never slides and positions
#     never shift.
#   - With prompt length >= prefix_length, the bidirectional prefix
#     (first 512 tokens) is fixed, so prefix hidden states never
#     change. New tokens lie after the prefix, where the encoder
#     branch outputs exactly zero, so only causal attention is
#     needed for them.
#
# Prompts shorter than prefix_length would make the prefix grow with
# every token, so they fall back to the slow generate_tokens().
# ============================================================

import torch
import torch.nn.functional as F


def _split_heads(t, n_heads, head_dim):
    B, T, _ = t.shape
    return t.view(B, T, n_heads, head_dim).transpose(1, 2)


@torch.no_grad()
def _block_prefill(block, x, prefix_length):
    """Full forward of one HybridTransformerBlock; also returns causal K/V."""

    attn = block.causal_attention

    encoder_output = block.encoder_attention(
        block.encoder_norm(x),
        prefix_length,
    )

    h = block.causal_norm(x)
    B, T, D = h.shape

    q = _split_heads(attn.q_proj(h), attn.n_heads, attn.head_dim)
    k = _split_heads(attn.k_proj(h), attn.n_heads, attn.head_dim)
    v = _split_heads(attn.v_proj(h), attn.n_heads, attn.head_dim)

    o = F.scaled_dot_product_attention(q, k, v, is_causal=True)
    o = attn.out_proj(o.transpose(1, 2).reshape(B, T, D))

    fused, _ = block.hybrid_fusion(encoder_output, o)

    x = x + fused
    x = x + block.ffn(block.ffn_norm(x))

    return x, k, v


@torch.no_grad()
def _block_decode(block, x, k_cache, v_cache, write_pos, attn_mask):
    """
    One new token per sequence.

    x         : [B, 1, D]
    k_cache   : [B, H, BLOCK_SIZE, head_dim]   (updated in place)
    write_pos : [B]  position of the new token in each sequence
    attn_mask : [B, 1, 1, BLOCK_SIZE] bool, True = may attend
    """

    attn = block.causal_attention
    B, _, D = x.shape

    h = block.causal_norm(x)

    q = _split_heads(attn.q_proj(h), attn.n_heads, attn.head_dim)
    k = _split_heads(attn.k_proj(h), attn.n_heads, attn.head_dim)
    v = _split_heads(attn.v_proj(h), attn.n_heads, attn.head_dim)

    rows = torch.arange(B, device=x.device)

    k_cache[rows, :, write_pos] = k[:, :, 0].to(k_cache.dtype)
    v_cache[rows, :, write_pos] = v[:, :, 0].to(v_cache.dtype)

    o = F.scaled_dot_product_attention(
        q,
        k_cache.to(q.dtype),
        v_cache.to(q.dtype),
        attn_mask=attn_mask,
    )

    o = attn.out_proj(o.transpose(1, 2).reshape(B, 1, D))

    # New tokens are after the prefix: encoder branch output is zero.
    fused, _ = block.hybrid_fusion(torch.zeros_like(o), o)

    x = x + fused
    x = x + block.ffn(block.ffn_norm(x))

    return x


def _apply_repetition_penalty(logits, seq, lens, penalty, window):
    """Batched version of the penalty used in generate_tokens()."""

    if penalty is None or penalty == 1.0:
        return logits

    B, T = seq.shape
    positions = torch.arange(T, device=seq.device)[None, :]

    valid = (
        (positions < lens[:, None])
        & (positions >= (lens - window)[:, None])
    )

    presence = torch.zeros_like(logits)
    presence.scatter_reduce_(1, seq, valid.to(logits.dtype), reduce="amax")
    presence = presence > 0

    penalised = torch.where(logits > 0, logits / penalty, logits * penalty)

    return torch.where(presence, penalised, logits)


def _sample_next(logits, temperature, top_k, top_p):
    """Batched greedy / temperature / top-k / top-p sampling."""

    if temperature is None or temperature <= 0:
        return torch.argmax(logits, dim=-1)

    logits = logits / temperature

    if top_k is not None and top_k > 0:
        k = min(int(top_k), logits.size(-1))
        values, indices = torch.topk(logits, k=k, dim=-1)
        filtered = torch.full_like(logits, float("-inf"))
        filtered.scatter_(-1, indices, values)
        logits = filtered

    if top_p is not None and 0.0 < top_p < 1.0:
        sorted_logits, sorted_indices = torch.sort(logits, descending=True, dim=-1)
        cumulative = torch.cumsum(torch.softmax(sorted_logits, dim=-1), dim=-1)
        remove = cumulative > top_p
        remove[:, 1:] = remove[:, :-1].clone()
        remove[:, 0] = False
        sorted_logits = sorted_logits.masked_fill(remove, float("-inf"))
        logits = torch.full_like(logits, float("-inf"))
        logits.scatter_(-1, sorted_indices, sorted_logits)

    probs = torch.softmax(logits, dim=-1)

    if not torch.isfinite(probs).all():
        raise RuntimeError("Non-finite probabilities encountered during generation.")

    return torch.multinomial(probs, num_samples=1).squeeze(-1)


def _autocast():
    if AMP_ENABLED and device.type == "cuda":
        return torch.autocast(device_type="cuda", dtype=AMP_DTYPE)
    return torch.autocast(device_type="cpu", enabled=False)


@torch.no_grad()
def generate_batch_fast(
    model,
    prompt_id_lists,
    max_new_tokens=512,
    temperature=0.8,
    top_k=50,
    top_p=0.95,
    repetition_penalty=1.0,
    repetition_window=256,
):
    """
    KV-cached batched generation.

    prompt_id_lists : list of token-id lists, each at least
                      CONFIG.prefix_length tokens long.

    Returns a list of lists: ONLY the new token ids per prompt,
    cut before EOS.
    """

    was_training = model.training
    model.eval()

    try:

        P = model.config.prefix_length
        max_prompt = BLOCK_SIZE - max_new_tokens

        if max_prompt < P:
            raise ValueError(
                f"max_new_tokens={max_new_tokens} leaves room for only "
                f"{max_prompt} prompt tokens, less than prefix_length={P}."
            )

        prompts = [list(p)[-max_prompt:] for p in prompt_id_lists]

        for p in prompts:
            if len(p) < P:
                raise ValueError(
                    f"Prompt of {len(p)} tokens is shorter than "
                    f"prefix_length={P}; use generate_tokens() instead."
                )

        dev = next(model.parameters()).device
        B = len(prompts)
        n_layers = len(model.blocks)
        eos_id = getattr(tokenizer, "eos_id", None)
        pad_fill = eos_id if eos_id is not None else 0

        lens = torch.tensor([len(p) for p in prompts], device=dev)
        seq = torch.full((B, BLOCK_SIZE), pad_fill, dtype=torch.long, device=dev)

        k_caches = [None] * n_layers
        v_caches = [None] * n_layers
        last_logits = None

        # --------------------------------------------------------
        # PREFILL: run each prompt once and fill the caches
        # --------------------------------------------------------

        for b, p in enumerate(prompts):

            ids = torch.tensor(p, dtype=torch.long, device=dev)[None, :]
            T = ids.size(1)
            seq[b, :T] = ids[0]

            positions = torch.arange(T, device=dev)

            with _autocast():

                x = model.token_embedding(ids) + model.position_embedding(positions)

                for li, block in enumerate(model.blocks):

                    x, k, v = _block_prefill(block, x, P)

                    if k_caches[li] is None:
                        shape = (B, k.size(1), BLOCK_SIZE, k.size(3))
                        k_caches[li] = torch.zeros(shape, dtype=k.dtype, device=dev)
                        v_caches[li] = torch.zeros(shape, dtype=v.dtype, device=dev)

                    k_caches[li][b, :, :T] = k[0]
                    v_caches[li][b, :, :T] = v[0]

                logits = model.lm_head(model.final_norm(x[:, -1:, :]))[:, 0].float()

            if last_logits is None:
                last_logits = torch.empty(B, logits.size(-1), device=dev)

            last_logits[b] = logits[0]

        # --------------------------------------------------------
        # DECODE: one token per sequence per step
        # --------------------------------------------------------

        new_tokens = torch.full((B, max_new_tokens), pad_fill, dtype=torch.long, device=dev)
        done = torch.zeros(B, dtype=torch.bool, device=dev)
        rows = torch.arange(B, device=dev)
        key_positions = torch.arange(BLOCK_SIZE, device=dev)[None, :]

        steps_taken = 0

        for step in range(max_new_tokens):

            logits = _apply_repetition_penalty(
                last_logits, seq, lens, repetition_penalty, repetition_window
            )

            next_token = _sample_next(logits, temperature, top_k, top_p)
            next_token = torch.where(done, torch.full_like(next_token, pad_fill), next_token)

            new_tokens[:, step] = next_token
            seq[rows, lens] = next_token
            steps_taken = step + 1

            if eos_id is not None:
                done = done | (next_token == eos_id)

            if bool(done.all()) or step == max_new_tokens - 1:
                break

            write_pos = lens.clone()
            attn_mask = (key_positions <= write_pos[:, None])[:, None, None, :]

            with _autocast():

                x = (
                    model.token_embedding(next_token[:, None])
                    + model.position_embedding(write_pos[:, None])
                )

                for li, block in enumerate(model.blocks):
                    x = _block_decode(
                        block, x, k_caches[li], v_caches[li], write_pos, attn_mask
                    )

                last_logits = model.lm_head(model.final_norm(x))[:, 0].float()

            lens = lens + 1

        # --------------------------------------------------------
        # Collect outputs, cut at EOS
        # --------------------------------------------------------

        results = []

        for row in new_tokens[:, :steps_taken].tolist():
            if eos_id is not None and eos_id in row:
                row = row[: row.index(eos_id)]
            results.append(row)

        return results

    finally:
        if was_training:
            model.train()


def generate_texts(
    model,
    prompt_id_lists,
    max_new_tokens=GEN_MAX_NEW_TOKENS,
    batch_size=None,
    **sampling,
):
    """
    Generate continuations (as text) for many prompts.

    Prompts >= prefix_length tokens use the fast batched path;
    shorter prompts fall back to the slow generate_tokens().
    Returns texts in the same order as prompt_id_lists.
    """

    if batch_size is None:
        batch_size = GEN_BATCH_SIZE

    texts = [None] * len(prompt_id_lists)

    fast_idx = [
        i for i, p in enumerate(prompt_id_lists)
        if len(p) >= CONFIG.prefix_length
    ]
    slow_idx = [
        i for i, p in enumerate(prompt_id_lists)
        if len(p) < CONFIG.prefix_length
    ]

    for start in range(0, len(fast_idx), batch_size):

        chunk = fast_idx[start:start + batch_size]

        outputs = generate_batch_fast(
            model,
            [prompt_id_lists[i] for i in chunk],
            max_new_tokens=max_new_tokens,
            **sampling,
        )

        for i, ids in zip(chunk, outputs):
            texts[i] = tokenizer.decode(ids)

    for i in slow_idx:

        input_tensor = torch.tensor(
            prompt_id_lists[i][-BLOCK_SIZE:], dtype=torch.long, device=device
        ).unsqueeze(0)

        new_ids = generate_tokens(
            model=model,
            input_ids=input_tensor,
            max_new_tokens=max_new_tokens,
            **sampling,
        )

        texts[i] = decode_continuation(new_ids)

    return texts


print("✓ generate_batch_fast() and generate_texts() ready")
print("  batch size:", GEN_BATCH_SIZE)


✓ generate_batch_fast() and generate_texts() ready
  batch size: 16


In [45]:
# ============================================================
# CELL 35c — CHECK FAST GENERATOR AGAINST THE SLOW ONE
#
# Greedy decoding with the real model on 3 test prompts.
# The two should agree; in fp16 they may diverge after a few
# dozen tokens because of tiny rounding differences, which is fine.
# Also shows the speedup.
# ============================================================

import time
import torch

CHECK_TOKENS = 64
# first 3 test prompts long enough for the fast path
check_items, check_prompts = [], []

for eid, ex in test_data.items():
    ids = tokenizer.encode(ex["text"])
    if len(ids) >= CONFIG.prefix_length:
        check_items.append((eid, ex))
        check_prompts.append(ids[-(BLOCK_SIZE - CHECK_TOKENS):])
    if len(check_items) == 3:
        break

greedy = dict(
    temperature=0,
    top_k=None,
    top_p=None,
    repetition_penalty=GEN_REPETITION_PENALTY,
    repetition_window=GEN_REPETITION_WINDOW,
)

# ---------------- slow ----------------
if device.type == "cuda":
    torch.cuda.synchronize()
t0 = time.time()

slow_outputs = []
for p in check_prompts:
    new = generate_tokens(
        model,
        torch.tensor(p, dtype=torch.long, device=device).unsqueeze(0),
        max_new_tokens=CHECK_TOKENS,
        **greedy,
    )[0].tolist()
    if tokenizer.eos_id in new:
        new = new[: new.index(tokenizer.eos_id)]
    slow_outputs.append(new)

if device.type == "cuda":
    torch.cuda.synchronize()
slow_time = time.time() - t0

# ---------------- fast ----------------
t0 = time.time()
fast_outputs = generate_batch_fast(
    model, check_prompts, max_new_tokens=CHECK_TOKENS, **greedy
)
if device.type == "cuda":
    torch.cuda.synchronize()
fast_time = time.time() - t0

print("=" * 70)
print("FAST vs SLOW GENERATOR")
print("=" * 70)

for (eid, _), a, b in zip(check_items, slow_outputs, fast_outputs):
    same_prefix = 0
    for x, y in zip(a, b):
        if x != y:
            break
        same_prefix += 1
    print(f"ID {eid}: identical for the first {same_prefix} / {max(len(a), len(b))} tokens")

print(f"\nSlow: {slow_time:.1f} s   Fast: {fast_time:.1f} s   "
      f"Speedup: {slow_time / max(fast_time, 1e-9):.1f}x")
print("(Batching more examples increases the speedup further.)")


FAST vs SLOW GENERATOR
ID 1: identical for the first 64 / 64 tokens
ID 2: identical for the first 64 / 64 tokens
ID 3: identical for the first 64 / 64 tokens

Slow: 9.4 s   Fast: 1.6 s   Speedup: 6.0x
(Batching more examples increases the speedup further.)


In [46]:
# ============================================================
# CELL 36 — PREVIEW GENERATION ON 10 TEST EXAMPLES
#
# Uses exactly the same generator and settings as the full run,
# so you can tune GEN_REPETITION_PENALTY / GEN_TEMPERATURE here first.
# ============================================================

import time

NUM_EXAMPLES = 10
PREVIEW_TOKENS = 256
SHOW_PROMPT_CHARS = 600   # only show the end of each prompt

preview_items = list(test_data.items())[:NUM_EXAMPLES]

preview_prompts = [
    tokenizer.encode(ex["text"]) for _, ex in preview_items
]

t0 = time.time()

preview_texts = generate_texts(
    model,
    preview_prompts,
    max_new_tokens=PREVIEW_TOKENS,
    temperature=GEN_TEMPERATURE,
    top_k=GEN_TOP_K,
    top_p=GEN_TOP_P,
    repetition_penalty=GEN_REPETITION_PENALTY,
    repetition_window=GEN_REPETITION_WINDOW,
)

elapsed = time.time() - t0

for count, ((example_id, example), continuation) in enumerate(
    zip(preview_items, preview_texts), start=1
):

    print("\n" + "=" * 80)
    print(f"EXAMPLE {count}/{len(preview_items)}   ID: {example_id}")
    print("=" * 80)

    print("\nPROMPT (last part)")
    print("-" * 80)
    print("..." + example["text"][-SHOW_PROMPT_CHARS:])

    print("\nGENERATED CONTINUATION")
    print("-" * 80)
    print(continuation)

print("\n" + "=" * 80)
print(f"10-EXAMPLE PREVIEW COMPLETE in {elapsed:.1f} s")
print("=" * 80)


[output removed before publishing: this cell printed raw dataset / test text]


In [47]:
# ============================================================
# CELL 37 — RESUMABLE GENERATION STATE
# ============================================================

import json

# Set to False to throw away previous progress and start over.
RESUME_GENERATION = False


if RESUME_GENERATION and PREDICTION_PROGRESS_PATH.exists():

    with open(PREDICTION_PROGRESS_PATH, "r", encoding="utf-8") as f:
        predictions = json.load(f)

    print("Loaded previous predictions:", len(predictions))

else:

    predictions = {}
    print("Starting generation from scratch.")


if RESUME_GENERATION and FAILURE_LOG_PATH.exists():

    with open(FAILURE_LOG_PATH, "r", encoding="utf-8") as f:
        failures = json.load(f)

else:

    failures = {}


# Drop any empty predictions left over from the old buggy run
# (the old slicing bug produced empty strings).
empty_ids = [
    k for k, v in predictions.items()
    if (v if isinstance(v, str) else v.get("text", "")).strip() == ""
]

for k in empty_ids:
    del predictions[k]

if empty_ids:
    print(f"Removed {len(empty_ids)} empty predictions; they will be regenerated.")


print("\n" + "=" * 70)
print("GENERATION STATUS")
print("=" * 70)
print("Total examples     :", len(test_data))
print("Completed          :", len(predictions))
print("Previous failures  :", len(failures))
print("Remaining          :", len(test_data) - len(predictions))
print("=" * 70)


Starting generation from scratch.

GENERATION STATUS
Total examples     : 1000
Completed          : 0
Previous failures  : 0
Remaining          : 1000


In [48]:
# ============================================================
# CELL 38 — FULL RESUMABLE TEST GENERATION (FAST, BATCHED)
#
# Safe to interrupt: progress is saved after every batch AND when
# the cell stops for any reason. Re-run cell 37 and this cell to
# continue.
# ============================================================

import json
import time
import torch


def _atomic_json_dump(obj, path):
    tmp = path.with_suffix(".tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)
    tmp.replace(path)


def save_generation_progress():
    _atomic_json_dump(predictions, PREDICTION_PROGRESS_PATH)
    _atomic_json_dump(failures, FAILURE_LOG_PATH)


SAMPLING = dict(
    temperature=GEN_TEMPERATURE,
    top_k=GEN_TOP_K,
    top_p=GEN_TOP_P,
    repetition_penalty=GEN_REPETITION_PENALTY,
    repetition_window=GEN_REPETITION_WINDOW,
)

model.eval()

# ------------------------------------------------------------
# Collect pending examples
# ------------------------------------------------------------

pending = []

for example_id, example in test_data.items():

    example_id = str(example_id)

    if example_id in predictions:
        continue

    prompt = example.get("text", "") if isinstance(example, dict) else example

    if not isinstance(prompt, str) or len(prompt.strip()) == 0:
        failures[example_id] = {"error": "Empty or non-string prompt."}
        continue

    prompt_ids = tokenizer.encode(prompt)

    if len(prompt_ids) == 0:
        failures[example_id] = {"error": "Tokenizer produced zero tokens."}
        continue

    pending.append((example_id, prompt_ids))

print("Pending examples:", len(pending))
print("Batch size      :", GEN_BATCH_SIZE)


def generate_one_with_retries(prompt_ids):
    """Fallback for a single example (used when a batch fails)."""

    last_error = None

    for attempt in range(MAX_RETRIES + 1):
        try:
            return generate_texts(
                model, [prompt_ids],
                max_new_tokens=GEN_MAX_NEW_TOKENS,
                batch_size=1,
                **SAMPLING,
            )[0], None
        except KeyboardInterrupt:
            raise
        except Exception as e:
            last_error = repr(e)
            if device.type == "cuda":
                torch.cuda.empty_cache()

    return None, last_error


generation_start = time.time()
done_this_session = 0

try:

    for start in range(0, len(pending), GEN_BATCH_SIZE):

        batch = pending[start:start + GEN_BATCH_SIZE]
        batch_ids = [eid for eid, _ in batch]
        batch_prompts = [p for _, p in batch]

        # ----------------------------------------------------
        # Try the whole batch at once
        # ----------------------------------------------------

        try:

            texts = generate_texts(
                model,
                batch_prompts,
                max_new_tokens=GEN_MAX_NEW_TOKENS,
                batch_size=GEN_BATCH_SIZE,
                **SAMPLING,
            )

            for eid, t in zip(batch_ids, texts):
                predictions[eid] = t
                failures.pop(eid, None)

        except KeyboardInterrupt:
            raise

        except Exception as e:

            # e.g. CUDA out-of-memory: redo this batch one by one
            print(f"⚠ Batch failed ({e!r}); retrying examples one by one.")

            if device.type == "cuda":
                torch.cuda.empty_cache()

            for eid, p in batch:
                t, err = generate_one_with_retries(p)
                if t is not None:
                    predictions[eid] = t
                    failures.pop(eid, None)
                else:
                    failures[eid] = {"error": err}

        done_this_session += len(batch)

        # ----------------------------------------------------
        # Save + progress report after every batch
        # ----------------------------------------------------

        save_generation_progress()

        elapsed = time.time() - generation_start
        per_example = elapsed / done_this_session
        remaining = len(pending) - done_this_session

        print(
            f"[{done_this_session}/{len(pending)}] "
            f"completed={len(predictions)} "
            f"failed={len(failures)} "
            f"elapsed={elapsed / 60:.1f} min "
            f"({per_example:.1f} s/example) "
            f"ETA={remaining * per_example / 60:.1f} min"
        )

finally:

    save_generation_progress()

    print("\n" + "=" * 70)
    print("GENERATION STOPPED / FINISHED — PROGRESS SAVED")
    print("=" * 70)
    print("Completed :", len(predictions), "/", len(test_data))
    print("Failed    :", len(failures))
    print("Progress  :", PREDICTION_PROGRESS_PATH)


Pending examples: 1000
Batch size      : 16
[16/1000] completed=16 failed=0 elapsed=0.2 min (0.8 s/example) ETA=12.3 min
[32/1000] completed=32 failed=0 elapsed=0.4 min (0.7 s/example) ETA=12.0 min
[48/1000] completed=48 failed=0 elapsed=0.6 min (0.7 s/example) ETA=11.8 min
[64/1000] completed=64 failed=0 elapsed=0.8 min (0.7 s/example) ETA=11.6 min
[80/1000] completed=80 failed=0 elapsed=1.0 min (0.7 s/example) ETA=11.4 min
[96/1000] completed=96 failed=0 elapsed=1.2 min (0.7 s/example) ETA=11.3 min
[112/1000] completed=112 failed=0 elapsed=1.4 min (0.7 s/example) ETA=11.1 min
[128/1000] completed=128 failed=0 elapsed=1.6 min (0.7 s/example) ETA=10.9 min
[144/1000] completed=144 failed=0 elapsed=1.8 min (0.7 s/example) ETA=10.7 min
[160/1000] completed=160 failed=0 elapsed=2.0 min (0.8 s/example) ETA=10.5 min
[176/1000] completed=176 failed=0 elapsed=2.2 min (0.8 s/example) ETA=10.3 min
[192/1000] completed=192 failed=0 elapsed=2.4 min (0.8 s/example) ETA=10.1 min
[208/1000] completed

In [49]:
# ============================================================
# CREATE FINAL prediction.json
# ============================================================

import json
from pathlib import Path


# ============================================================
# FINAL OUTPUT PATH
# ============================================================

FINAL_PREDICTION_PATH = Path(
    "/kaggle/working/prediction.json"
)


# ============================================================
# SAFETY CHECKS
# ============================================================

if len(failures) > 0:

    raise RuntimeError(
        f"Cannot create final prediction.json: "
        f"{len(failures)} test examples still failed.\n"
        f"Please regenerate the failed examples first."
    )


missing_ids = [
    str(test_id)
    for test_id in test_data.keys()
    if str(test_id) not in predictions
]


if len(missing_ids) > 0:

    raise RuntimeError(
        f"Cannot create final prediction.json: "
        f"{len(missing_ids)} test examples are missing."
    )


if len(predictions) != len(test_data):

    raise RuntimeError(
        f"Prediction count does not match test-data count.\n"
        f"Predictions: {len(predictions)}\n"
        f"Test data  : {len(test_data)}"
    )


# ============================================================
# BUILD EXACT SUBMISSION FORMAT
# ============================================================

final_predictions = {}

for test_id in test_data.keys():

    test_id = str(test_id)

    prediction = predictions[test_id]

    # --------------------------------------------------------
    # Our generation code stores prediction as a string:
    #
    # predictions[test_id] = "generated text..."
    #
    # Convert it into the required:
    #
    # {
    #     "text": "generated text..."
    # }
    # --------------------------------------------------------

    if isinstance(
        prediction,
        str
    ):

        final_predictions[test_id] = {
            "text": prediction
        }

    # --------------------------------------------------------
    # Also support the case where prediction is already:
    #
    # {"text": "..."}
    # --------------------------------------------------------

    elif isinstance(
        prediction,
        dict
    ):

        if "text" not in prediction:

            raise ValueError(
                f"Prediction for ID {test_id} "
                f"does not contain 'text'."
            )

        final_predictions[test_id] = {
            "text": prediction["text"]
        }

    else:

        raise TypeError(
            f"Unsupported prediction type for "
            f"{test_id}: {type(prediction)}"
        )


# ============================================================
# WRITE ATOMICALLY
# ============================================================

TEMP_PREDICTION_PATH = (
    FINAL_PREDICTION_PATH.with_suffix(
        ".tmp"
    )
)


with open(
    TEMP_PREDICTION_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        final_predictions,
        f,
        ensure_ascii=False,
        indent=2,
    )


# Replace old file only after successful write
TEMP_PREDICTION_PATH.replace(
    FINAL_PREDICTION_PATH
)


# ============================================================
# VERIFY FINAL FILE
# ============================================================

with open(
    FINAL_PREDICTION_PATH,
    "r",
    encoding="utf-8",
) as f:

    check = json.load(f)


assert len(check) == len(test_data)

for test_id in test_data:

    test_id = str(test_id)

    assert test_id in check
    assert isinstance(
        check[test_id],
        dict
    )
    assert "text" in check[test_id]


# ============================================================
# REPORT
# ============================================================

print("=" * 70)
print("FINAL prediction.json CREATED")
print("=" * 70)

print(
    "Path       :",
    FINAL_PREDICTION_PATH
)

print(
    "Examples   :",
    len(final_predictions)
)

print(
    "File size  :",
    f"{FINAL_PREDICTION_PATH.stat().st_size / 1024:.2f} KB"
)


# ============================================================
# SHOW FIRST EXAMPLE
# ============================================================

first_id = next(
    iter(final_predictions)
)

print()
print(
    "First prediction ID:",
    first_id
)

print(
    "\nFirst prediction:"
)

print(
    final_predictions[
        first_id
    ]["text"]
)

print(
    "\n✓ Final prediction.json is ready."
)

FINAL prediction.json CREATED
Path       : /kaggle/working/prediction.json
Examples   : 1000
File size  : 1128.59 KB

First prediction ID: 1

First prediction:




✓ Final prediction.json is ready.


In [50]:
# ============================================================
# CELL 39 — VERIFY FINAL prediction.json
# ============================================================

import json

if not FINAL_PREDICTION_PATH.exists():
    raise FileNotFoundError("prediction.json was not created.")

with open(FINAL_PREDICTION_PATH, "r", encoding="utf-8") as f:
    final_predictions = json.load(f)

print("=" * 70)
print("PREDICTION.JSON VERIFICATION")
print("=" * 70)

print("Total predictions:", len(final_predictions))
print("Total test data  :", len(test_data))
print("Failures         :", len(failures))

missing_ids = [
    str(example_id)
    for example_id in test_data
    if str(example_id) not in final_predictions
]

bad_format = [
    k for k, v in final_predictions.items()
    if not (isinstance(v, dict) and isinstance(v.get("text"), str))
]

empty_texts = [
    k for k, v in final_predictions.items()
    if isinstance(v, dict) and v.get("text", "").strip() == ""
]

eos_leaks = [
    k for k, v in final_predictions.items()
    if isinstance(v, dict) and "<eos>" in v.get("text", "")
]

print("Missing IDs      :", len(missing_ids))
print("Bad format       :", len(bad_format))
print("Empty texts      :", len(empty_texts))
print("Texts with <eos> :", len(eos_leaks))

if final_predictions:
    first_prediction_id = next(iter(final_predictions))
    print("\nFirst prediction ID:", first_prediction_id)
    print("\nPrediction:")
    print(final_predictions[first_prediction_id]["text"][:1000])

assert isinstance(final_predictions, dict)
assert len(missing_ids) == 0
assert len(bad_format) == 0
assert len(eos_leaks) == 0

if empty_texts:
    print(f"\n⚠ {len(empty_texts)} predictions are empty "
          "(model emitted EOS immediately). Consider regenerating them.")

print("\n✓ prediction.json exists")
print("✓ All test IDs have predictions in {\"text\": ...} format")
print("=" * 70)


[output removed before publishing: this cell printed raw dataset / test text]
